This notebook is for Core persona extraction:

clustering, archetypes, sentiment, behavioral profiles

In [8]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

In [9]:
import pandas as pd
import numpy as np

print("Environment working")

Environment working


In [10]:
import pandas as pd
import numpy as np

# File paths
review_path = "../data/raw/yelp/yelp_academic_dataset_review.json"
business_path = "../data/raw/yelp/yelp_academic_dataset_business.json"
user_path = "../data/raw/yelp/yelp_academic_dataset_user.json"
print("Paths loaded successfully")

Paths loaded successfully


In [11]:
from src.personas.archetypes import *

In [12]:
from src.personas.value_system import *

In [13]:
from src.personas.emotional_drift import *

In [14]:
# STEP 3
# Loading dataframes
# For initial exploration, we will load a sample of the reviews and users datasets to avoid memory issues, while loading the entire business dataset to analyze restaurant-related information.
# This approach allows us to get a sense of the structure and content of the datasets without overwhelming our system's memory, while still providing us with enough data to perform meaningful analysis on restaurant reviews and business information.

reviews_sample = pd.read_json(
    review_path,
    lines=True,
    nrows=5000
)

business_df = pd.read_json(
    business_path,
    lines=True,
)

users_sample = pd.read_json(
    user_path,
    lines=True,
    nrows=5000
)

print("Reviews shape:", reviews_sample.shape)
print("Business dataset shape:")
print(business_df.shape)
print("Users shape:", users_sample.shape)

Reviews shape: (5000, 9)
Business dataset shape:
(150346, 14)
Users shape: (5000, 22)


In [15]:
# STEP 4
# Inspecting columns of each dataset
# This will help us understand the structure of the data and identify key features for analysis.

print("REVIEWS COLUMNS")
print(reviews_sample.columns)

print("\nBUSINESS COLUMNS")
print(business_df.columns)

print("\nUSER COLUMNS")
print(users_sample.columns)

REVIEWS COLUMNS
Index(['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny',
       'cool', 'text', 'date'],
      dtype='str')

BUSINESS COLUMNS
Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'attributes', 'categories', 'hours'],
      dtype='str')

USER COLUMNS
Index(['user_id', 'name', 'review_count', 'yelping_since', 'useful', 'funny',
       'cool', 'elite', 'friends', 'fans', 'average_stars', 'compliment_hot',
       'compliment_more', 'compliment_profile', 'compliment_cute',
       'compliment_list', 'compliment_note', 'compliment_plain',
       'compliment_cool', 'compliment_funny', 'compliment_writer',
       'compliment_photos'],
      dtype='str')


In [16]:
# STEP 5
# Displaying first few rows of each dataset to get a sense of the data
reviews_sample.head(2)

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18


In Reviews, there are important fields like: user_id, business_id, stars, text, date

These serves as our behavioral signal layer.

In [17]:
# STEP 6
# Displaying first few rows of business dataset

business_df.head(2)

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."


In Businesses

Fields like: categories,city, attributes,stars

This serves as our contextual environment layer.

In [18]:

# STEP 7
# Displaying first few rows of users dataset

users_sample.head(2)

,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,...,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,...,264,184,157,251,1847,7054,3131,3131,1521,1946


In Users fields like: review_count, average_stars, fans

These serves as our behavioral metadata layer.

STEP 8 
Filtering Restaurant Businesses

We only want businesses related to: restaurants, food, cafes, bars because this domain contains rich emotional/social behavior.

In [19]:
# Keep only businesses with restaurant-related categories
# This will help us focus our analysis on the restaurant industry, which is a major part of Yelp's business and user interactions.
# filters businesses whose categories contain: Restaurant, Food, Coffee, Cafe, Bar

restaurant_businesses = business_df[
    business_df["categories"]
    .fillna("")
    .str.contains(
        "Restaurant|Food|Coffee|Cafe|Bar",
        case=False,
        regex=True
    )
]

print("Restaurant businesses:")
print(restaurant_businesses.shape)

Restaurant businesses:
(68696, 14)


In [20]:
# STEP 9
# Inspect Restaurant businesses

restaurant_businesses[
    ["business_id", "name", "categories", "city"]
].head(10)

,business_id,name,categories,city
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",Philadelphia
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,"Brewpubs, Breweries, Food",Green Lane
5,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...",Ashland City
8,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,"Pubs, Restaurants, Italian, Bars, American (Tr...",Affton
9,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,"Ice Cream & Frozen Yogurt, Fast Food, Burgers,...",Nashville
11,eEOYSgkmpB90uNA7lDOMRA,Vietnamese Food Truck,"Vietnamese, Food, Restaurants, Food Trucks",Tampa Bay
12,il_Ro8jwPlHresjw9EGmBg,Denny's,"American (Traditional), Restaurants, Diners, B...",Indianapolis
14,0bPLkL0QhhPO5kt1_EXmNQ,Zio's Italian Market,"Food, Delis, Italian, Bakeries, Restaurants",Largo
15,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,"Sushi Bars, Restaurants, Japanese",Philadelphia
19,ROeacJQwBeh05Rqg7F6TCg,BAP,"Korean, Restaurants",Philadelphia


In [21]:
# STEP 10
# Extract unique restaurant business IDs

restaurant_ids = set(
    restaurant_businesses["business_id"]
)

print("Number of restaurant IDs:")
print(len(restaurant_ids))

Number of restaurant IDs:
68696


In [22]:
# STEP 11
# Filter reviews to include only those related to the restaurant businesses identified above. 
# This will allow us to analyze user feedback specifically for restaurants, which is crucial for understanding customer satisfaction and business performance in this sector.
# This makes personas cleaner, preferences clearer, recommendations better

restaurant_reviews = reviews_sample[
    reviews_sample["business_id"].isin(
        restaurant_businesses["business_id"]
    )
]

print("Restaurant reviews shape:")
print(restaurant_reviews.shape)

Restaurant reviews shape:
(3987, 9)


In [23]:
# STEP 12
#  Inspect restaurant reviews
# This will give us insights into the types of feedback customers are providing for restaurants, which can inform our analysis of customer satisfaction and business performance in this sector.

restaurant_reviews[
    ["user_id", "business_id", "stars", "text"]
].head(5)

,user_id,business_id,stars,text
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,"If you decide to eat here, just be aware it is..."
2,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,Family diner. Had the buffet. Eclectic assortm...
3,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,"Wow! Yummy, different, delicious. Our favo..."
4,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,Cute interior and owner (?) gave us tour of up...
5,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1,I am a long term frequent customer of this est...


Finally, I am looking at real behavioral data. I can now see:

emotions, complaints, enthusiasm, sarcasm, values, judgments

This is a raw psychological signal.

STEP 13: Find Sweet-Spot Users

Now we identify behaviorally rich users.

In [24]:
# Count reviews per user
# This will help us identify active users and understand the distribution of reviews among users, which can inform our analysis of user behavior and preferences in the restaurant sector.

user_review_counts = (
    restaurant_reviews
    .groupby("user_id")
    .size()
    .reset_index(name="review_count")
)

# Keep users with 15–50 reviews

sweet_spot_users = user_review_counts[
    (user_review_counts["review_count"] >= 15) &
    (user_review_counts["review_count"] <= 50)
]

print("Sweet-spot users:")
print(sweet_spot_users.shape)

Sweet-spot users:
(0, 2)


In [25]:
# STEP 13B
# Inspect sweet-spot users
sweet_spot_users.head(10)

,user_id,review_count


In [26]:
# STEP 14
# Read review data in chunks
# This approach allows us to process large datasets without running into memory issues, enabling us to filter and analyze reviews related to restaurants efficiently.
# By reading the review data in chunks, we can handle the large size of the dataset while still extracting relevant information for our analysis of restaurant reviews.

chunk_size = 100000

restaurant_review_chunks = []

for chunk in pd.read_json(
    review_path,
    lines=True,
    chunksize=chunk_size
):
    
    filtered_chunk = chunk[
        chunk["business_id"].isin(restaurant_ids)
    ]
    
    restaurant_review_chunks.append(filtered_chunk)

    print(
        f"Processed chunk with "
        f"{len(filtered_chunk)} restaurant reviews"
    )

Processed chunk with 80537 restaurant reviews
Processed chunk with 80648 restaurant reviews
Processed chunk with 79453 restaurant reviews
Processed chunk with 75590 restaurant reviews
Processed chunk with 71692 restaurant reviews
Processed chunk with 70100 restaurant reviews
Processed chunk with 68485 restaurant reviews
Processed chunk with 79838 restaurant reviews
Processed chunk with 81158 restaurant reviews
Processed chunk with 80883 restaurant reviews
Processed chunk with 77279 restaurant reviews
Processed chunk with 72916 restaurant reviews
Processed chunk with 70352 restaurant reviews
Processed chunk with 68087 restaurant reviews
Processed chunk with 78749 restaurant reviews
Processed chunk with 80778 restaurant reviews
Processed chunk with 80846 restaurant reviews
Processed chunk with 77760 restaurant reviews
Processed chunk with 73650 restaurant reviews
Processed chunk with 69105 restaurant reviews
Processed chunk with 69018 restaurant reviews
Processed chunk with 77708 restaur

In [27]:
# Step 15
# Concatenate all filtered chunks into a single DataFrame
# This will give us a complete dataset of restaurant reviews that we can use for further analysis, 
# such as sentiment analysis, user behavior analysis, and business performance evaluation in the restaurant sector.
# This is our real behavioral corpus.

restaurant_reviews = pd.concat(
    restaurant_review_chunks,
    ignore_index=True
)

print("Final restaurant reviews shape:")
print(restaurant_reviews.shape)

Final restaurant reviews shape:
(5259993, 9)


In [28]:
# STEP 16
# Count reviews per user
# This will help us identify users who have reviewed a moderate number of restaurants, which might be indicative of engaged reviewers.

user_review_counts = (
    restaurant_reviews
    .groupby("user_id")
    .size()
    .reset_index(name="review_count")
)

sweet_spot_users = user_review_counts[
    (user_review_counts["review_count"] >= 15) &
    (user_review_counts["review_count"] <= 50)
]

print("Sweet-spot users:")
print(sweet_spot_users.shape)

Sweet-spot users:
(40884, 2)


In [29]:
# STEP 17
# Curate a sample of sweet-spot users for further analysis, 
# ensuring we have a manageable number of users to work with while still capturing a representative subset of engaged reviewers.

curated_users = sweet_spot_users.sample(
    n=300,
    random_state=42
)

print(curated_users.shape)

(300, 2)


In [30]:
# Step 18
# Filter restaurant reviews to include only those from the curated sweet-spot users.
# This will allow us to focus our analysis on a specific subset of engaged users, which can provide more meaningful insights into user behavior and preferences in the restaurant sector.

curated_user_ids = set(
    curated_users["user_id"]
)

curated_reviews = restaurant_reviews[
    restaurant_reviews["user_id"]
    .isin(curated_user_ids)
]

print("Curated reviews shape:")
print(curated_reviews.shape)

Curated reviews shape:
(7413, 9)


This reveals a curated corpus of: psychologically rich users, restaurant behaviors, emotional language, preferences, review styles

Enough to build: personas, review simulation, recommendations, conversational reasoning

In [31]:
# STEP 19
# Transforming raw reviews into interpretable human behavioral traits.
# First, we organize and group reviews by user and business, then we can analyze patterns in the review text, star ratings, and other features to derive insights about user preferences, sentiment, and engagement with restaurants. This will help us understand the underlying behaviors and traits of users in the context of restaurant reviews.



user_histories = (
    curated_reviews
    .groupby("user_id")
    .agg({
        "text": list,
        "stars": list,
        "business_id": list,
        "date": list
    })
    .reset_index()
)

print("User histories shape:")
print(user_histories.shape)

User histories shape:
(300, 5)


In [32]:
# STEP 20
# Inspecting a sample user history to understand the structure of the data and the type of information we have for each user, which will help us in deriving behavioral traits and insights from their reviews.
# Change the index to inspect different users and their review histories.

sample_user = user_histories.iloc[70]

print("USER ID:")
print(sample_user["user_id"])

print("\nSTAR RATINGS:")
print(sample_user["stars"][:5])

print("\nFIRST REVIEW:")
print(sample_user["text"][0][:500])

USER ID:
Gcxm0XlnMIW0sUwiYgo4dA

STAR RATINGS:
[4, 5, 5, 4, 5]

FIRST REVIEW:
Went here for Valentine's day. Very crowded but our reservation was on time and server was good. Grilled seafood app was the highlight of the meal. Grilled scallops, calamari, shrimp and cherry tomatoes over arugula perfectly dressed. 5 stars.
Had filet and pescatore for entree. Filet was good but not amazing- 3 stars. Pescatore was delicious- 5 stars. Lobster slightly dry, but  calamari and scallops and garlic white wine sauce were more than enough.
Dessert was a hit and a miss. Chocolate souff


By changing the index, i notice behavioral traces of a real human.

Things i notice:

tone
complaints
enthusiasm
emotional intensity
writing style
priorities

This is where personas emerge.

In [33]:
# STEP 21 — Creating First Persona Features
# Here we are calculating basic features for each user based on their reviews, such as average rating, rating variance, review count, and average review length. 
# These features can help us understand user behavior and preferences in the context of restaurant reviews, which can be useful for building user personas and improving recommendation systems.

persona_features = (
    curated_reviews
    .groupby("user_id")
    .agg(
        avg_rating=("stars", "mean"),
        rating_variance=("stars", "std"),
        review_count=("stars", "count"),
        avg_review_length=(
            "text",
            lambda x: np.mean(
                x.str.len()
            )
        )
    )
    .reset_index()
)

print(persona_features.shape)

persona_features.head()

(300, 5)


,user_id,avg_rating,rating_variance,review_count,avg_review_length
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455


WHAT THESE FEATURES MEAN
  Feature	               Psychological Meaning
- avg_rating	           harsh vs lenient
- rating_variance	       emotional consistency
- review_count	           engagement
- avg_review_length	       verbosity/detail orientation

These are behavioral signals.

In [127]:
import pandas as pd

yelp_df = pd.read_csv(
    "../data/curated/restaurants/yelp_subset.csv"
)

yelp_df.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15


In [182]:
# BUILD PERSONA DATAFRAME
# FROM USER-LEVEL REVIEW BEHAVIOR

persona_df = (

    yelp_df

    .groupby("user_id")

    .agg({

        "stars": [
            "mean",
            "std",
            "count"
        ],

        "text": lambda x: (
            x.astype(str)
            .str.len()
            .mean()
        )
    })

)

persona_df.columns = [

    "avg_rating",
    "rating_variance",
    "review_count",
    "avg_review_length"
   
]

persona_df = (
    persona_df
    .reset_index()
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length
0,--4AjktZiHowEIBCMd4CZA,4.0,NaN,1,351.0
1,--vCeHrklS1DIep0QhorrA,4.0,NaN,1,121.0
2,-0KrCHEsOcjJ6N4k_k1A9A,4.0,NaN,1,335.0
3,-2MXx9Fk3IiCg2y559iI8Q,4.0,NaN,1,269.0
4,-4Pvmnz9Mej98xiz1w-FRQ,2.0,NaN,1,341.0


In [129]:
from textblob import TextBlob
from tqdm import tqdm

tqdm.pandas()

In [130]:
# Sentiment per review

curated_reviews["sentiment"] = (
    curated_reviews["text"]
    .progress_apply(
        lambda x: TextBlob(x).sentiment.polarity
    )
)

  0%|          | 0/7413 [00:00<?, ?it/s]

100%|██████████| 7413/7413 [00:07<00:00, 1044.18it/s]


In [150]:
# COMPUTE USER SENTIMENT

user_sentiments = []

for user_id in persona_df["user_id"]:

    user_reviews = yelp_df[
        yelp_df["user_id"] == user_id
    ]["text"].astype(str)

    sentiments = [

        TextBlob(
            review
        ).sentiment.polarity

        for review in user_reviews
    ]

    avg_sentiment = (
        np.mean(sentiments)
        if sentiments else 0
    )

    sentiment_variance = (
        np.std(sentiments)
        if sentiments else 0
    )

    user_sentiments.append({

        "user_id":
            user_id,

        "avg_sentiment":
            avg_sentiment,

        "sentiment_variance":
            sentiment_variance
    })

sentiment_df = pd.DataFrame(
    user_sentiments
)

sentiment_df.head()

,user_id,avg_sentiment,sentiment_variance
0,--4AjktZiHowEIBCMd4CZA,0.383333,0.0
1,--vCeHrklS1DIep0QhorrA,0.290741,0.0
2,-0KrCHEsOcjJ6N4k_k1A9A,0.229167,0.0
3,-2MXx9Fk3IiCg2y559iI8Q,0.206250,0.0
4,-4Pvmnz9Mej98xiz1w-FRQ,0.153906,0.0


In [174]:
# STEP 23
# Aggregating User Sentiment

sentiment_features = (
    curated_reviews
    .groupby("user_id")
    .agg(
        avg_sentiment=("sentiment", "mean"),
        sentiment_variance=("sentiment", "std")
    )
    .reset_index()
)

sentiment_features.head()

,user_id,avg_sentiment,sentiment_variance
0,-EX1hrPRBqNkVavtMllTCA,0.138507,0.222461
1,-M7fUg7FrdGctKr5f_eMUQ,0.215294,0.162739
2,-WM58wLjtlHlR91xVfM1FQ,0.298114,0.206382
3,-qTtg1D3RidRa4cTB-ftwg,0.425713,0.245910
4,02H49g16MdRoZKoX6IEoFA,0.255776,0.355440


WHAT THESE MEAN
Feature	              Meaning
avg_sentiment	      positivity/negativity
sentiment_variance	  emotional stability

In [159]:
# STEP 24
# Merge Persona Features

persona_df = persona_features.merge(
    sentiment_features,
    on="user_id"
)

print(persona_df.shape)

persona_df.head()

(300, 7)


,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440


In [156]:
# STEP 25 — Interpret Personas

persona_df.describe()

,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance
count,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000
mean,3.884222,1.081557,24.710000,592.957436,0.256042,0.186791
std,0.604832,0.363819,9.462082,367.794262,0.087883,0.067733
min,1.591837,0.000000,15.000000,175.900000,-0.071841,0.066279
25%,3.523810,0.834853,17.000000,346.679412,0.198519,0.138404
50%,3.942810,1.095445,22.000000,499.306548,0.255502,0.172593
75%,4.315789,1.335010,31.000000,751.808527,0.314091,0.225523
max,5.000000,1.999557,50.000000,2534.300000,0.510412,0.430499


This uncovers human archetypes.

Examples:

angry critics
generous reviewers
emotional storytellers
concise pragmatists

In [144]:
# Harsh Users
# These users tend to give lower ratings on average, which may indicate a more critical perspective or higher standards when it comes to restaurant experiences.

persona_df.sort_values(
    by="avg_rating"
).head(5)

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment_x,sentiment_variance_x,avg_sentiment_y,sentiment_variance_y
20,3GKk2POn0VFznH_KuRX8UA,1.591837,1.153227,49,452.938776,0.034711,0.254318,NaN,NaN
7,0Tsu6-uhw_w9Z3W0Xlnazg,1.687500,1.138347,16,411.125000,0.093389,0.225517,NaN,NaN
199,gVvkw-bW7hcrs9P5xfwOhw,1.708333,1.366658,24,502.041667,0.031524,0.212126,NaN,NaN
179,cb5omh0nibWUYN2rws5rdg,1.850000,1.182103,20,529.450000,-0.071841,0.202567,NaN,NaN
225,mD-IgInk0o8pXrJI-P8pYA,2.266667,1.099784,15,297.333333,0.124339,0.143346,NaN,NaN


In [145]:
# Lenient Users
# These users tend to give higher ratings on average, which may indicate a more positive outlook or a tendency to be more forgiving in their reviews.

persona_df.sort_values(
    by="avg_rating",
    ascending=False
).head(5)

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment_x,sentiment_variance_x,avg_sentiment_y,sentiment_variance_y
212,jYOK8bu9lIxkVoKvwPC7ig,5.000000,0.000000,31,361.774194,0.382680,0.176973,NaN,NaN
81,I6x-ZBHeCNlMnmtfUVf5lg,5.000000,0.000000,24,176.208333,0.510412,0.158668,NaN,NaN
167,_m0Sxb2_kUFtic4wGwIGzw,4.960000,0.200000,25,249.040000,0.485915,0.160752,NaN,NaN
5,0650daOKAuufqymyOBe3cA,4.937500,0.250000,16,255.937500,0.478739,0.196268,NaN,NaN
23,4Z2lfaP3d3oOKmEfmc9PCw,4.933333,0.258199,15,828.200000,0.315061,0.154081,NaN,NaN


In [146]:
# Verbose Users
# These users tend to write longer reviews, which may indicate a higher level of engagement or a desire to provide more detailed feedback.

persona_df.sort_values(
    by="avg_review_length",
    ascending=False
).head(5)

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment_x,sentiment_variance_x,avg_sentiment_y,sentiment_variance_y
168,_p7aWe_YiAZW2m6RnmAMVA,3.325000,1.071484,40,2534.300000,0.121472,0.099103,NaN,NaN
247,q4Oz1c_OjGu_a8ZkP53Vnw,2.723404,1.378438,47,2529.531915,0.118203,0.093142,NaN,NaN
54,Do8oscF3LjCl-_pXrmgLXA,4.041667,0.954585,24,2228.416667,0.178727,0.066896,NaN,NaN
19,2xNYCJxZOrhVsrVMz8mlDQ,3.911765,0.933149,34,2181.441176,0.228747,0.066279,NaN,NaN
38,9VdyNBdQbaZTrFDmD49e7A,3.736842,1.097578,19,2119.368421,0.181577,0.117300,NaN,NaN


With these, we can:

characterize users
compare personalities
simulate tendencies
reason about preferences

Next, We will transform numeric persona signals into recognizable human archetypes.

Examples:

harsh critic
emotional foodie
soft-life explorer
concise pragmatist
luxury seeker

This becomes our Dynamic Cognitive Persona Layer.

WHY THIS MATTERS

The model is now grouping users by:

emotional style
harshness
verbosity
behavioral consistency

This is latent human behavior discovery.

In [183]:
print(type(persona_df))

<class 'pandas.DataFrame'>


In [184]:
# Check which columns actually exist in persona_df
print(persona_df.columns.tolist())

# Check if sentiment_features has the expected columns
print(sentiment_features.columns.tolist())

# Check the merge result
merged = persona_features.merge(sentiment_features, on="user_id", how="inner")  # or how="left"
print(merged.columns.tolist())

['user_id', 'avg_rating', 'rating_variance', 'review_count', 'avg_review_length']
['user_id', 'avg_sentiment', 'sentiment_variance']
['user_id', 'avg_rating', 'rating_variance', 'review_count', 'avg_review_length', 'avg_sentiment', 'sentiment_variance']


In [186]:
print(persona_df.columns.tolist())
print(sentiment_features.columns.tolist() if 'sentiment_features' in locals() else "sentiment_features not defined")

['user_id', 'avg_rating', 'rating_variance', 'review_count', 'avg_review_length']
['user_id', 'avg_sentiment', 'sentiment_variance']


In [187]:
# Re-run the merge and check columns
persona_df = persona_features.merge(sentiment_features, on="user_id", how="inner")
print("Columns after merge:", persona_df.columns.tolist())
print("Shape:", persona_df.shape)

Columns after merge: ['user_id', 'avg_rating', 'rating_variance', 'review_count', 'avg_review_length', 'avg_sentiment', 'sentiment_variance']
Shape: (300, 7)


In [188]:
# CLUSTERING AND ARCHETYPES

from src.personas.archetypes import (
    prepare_clustering_features,
    scale_features,
    run_kmeans_clustering,
    assign_archetypes
)


clustering_features = (
    prepare_clustering_features(
        persona_df
    )
)

scaled_features = (
    scale_features(
        clustering_features
    )
)

persona_df, kmeans = (
    run_kmeans_clustering(
        persona_df,
        scaled_features
    )
)

persona_df = assign_archetypes(
    persona_df
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer


In [189]:
persona_df.columns

Index(['user_id', 'avg_rating', 'rating_variance', 'review_count',
       'avg_review_length', 'avg_sentiment', 'sentiment_variance', 'cluster',
       'archetype'],
      dtype='str')

In [190]:
persona_df.to_csv(
    "../outputs/persona_dataset.csv",
    index=False
)

print("persona dataset saved")

persona dataset saved


In [44]:
# STEP 29 — Analyze Cluster Distribution. 
# This will help us understand how users are grouped into different personas based on their review behavior and sentiment, 
# which can inform our strategies for targeted marketing, personalized recommendations, and customer segmentation in the restaurant industry.

persona_df["cluster"].value_counts()

cluster
3    101
0     83
1     54
2     46
4     16
Name: count, dtype: int64

In [45]:
persona_df["archetype"].value_counts()

archetype
Emotional Storyteller      101
Warm Optimist               83
Reactive Reviewer           54
Harsh Critic                46
Deep Experience Analyst     16
Name: count, dtype: int64

In [46]:
# STEP 31C — Attach Structured Traits
# Add the full trait dictionaries.


persona_df["behavior_profile"] = (
    persona_df["cluster"]
    .apply(
        lambda x: behavioral_descriptors[x]["traits"]
    )
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic,"{'positivity': 'low', 'verbosity': 'high', 'em..."
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'..."
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'..."
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist,"{'positivity': 'high', 'verbosity': 'moderate'..."
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer,"{'positivity': 'mixed', 'verbosity': 'moderate..."


WHAT THE DATAFRAME NOW CONTAINS

Each user now has:

Column	                Meaning
avg_rating	            rating behavior
avg_review_length	    verbosity
avg_sentiment	        emotional tone
archetype	            human-readable identity
behavior_profile	    machine-readable cognition

In [47]:
# Qualitative Inspection of User Reviews by Archetype
for archetype in persona_df["archetype"].unique():

    print("\n" + "="*80)
    print(f"ARCHETYPE: {archetype}")
    print("="*80)

    sampled_users = (
        persona_df[
            persona_df["archetype"] == archetype
        ]
        .sample(2, random_state=42)   # fewer users
    )

    for _, user_row in sampled_users.iterrows():

        sample_user_id = user_row["user_id"]

        print("\n" + "#"*60)
        print(f"USER ID: {sample_user_id}")
        print("#"*60)

        user_reviews = curated_reviews[
            curated_reviews["user_id"] == sample_user_id
        ]

        for i, (_, row) in enumerate(
            user_reviews.head(1).iterrows(),  # only 1 review
            start=1
        ):

            print("\n" + "-"*50)
            print(f"Review #{i}")
            print(f"Stars: {row['stars']}")
            print("-"*50)

            print(row["text"][:200].replace("\n", " ") + "...")
            print("\n")


ARCHETYPE: Harsh Critic

############################################################
USER ID: oEMXXNNZiYUllytcZLtbvw
############################################################

--------------------------------------------------
Review #1
Stars: 5
--------------------------------------------------
Oh yum! And clean!! Friendly and efficient, too - I've been there four times so far. The third time, my wife wanted green curry, but they'd featured it the day before. We were very disappointed, until...



############################################################
USER ID: Y-NI1hIn1AB4lP3bT-eSog
############################################################

--------------------------------------------------
Review #1
Stars: 5
--------------------------------------------------
I love local businesses.  I don't drink coffee, but I am obsessed with chai tea lattes and Feine makes a great one.  I love that the baristas offer you different milk options too. I've tried both the ...



ARCHETYP

In [48]:
# STEP 32 — Final Persona Summary
persona_df[
    ["user_id", "archetype"]
].head(10)

,user_id,archetype
0,-EX1hrPRBqNkVavtMllTCA,Harsh Critic
1,-M7fUg7FrdGctKr5f_eMUQ,Emotional Storyteller
2,-WM58wLjtlHlR91xVfM1FQ,Emotional Storyteller
3,-qTtg1D3RidRa4cTB-ftwg,Warm Optimist
4,02H49g16MdRoZKoX6IEoFA,Reactive Reviewer
5,0650daOKAuufqymyOBe3cA,Warm Optimist
6,06Yz-YYYa1U9PN37b6UniA,Emotional Storyteller
7,0Tsu6-uhw_w9Z3W0Xlnazg,Harsh Critic
8,0xJwTzZuWOac7ufeTioeig,Emotional Storyteller
9,175DuOm7IPiEy5f1KG9esA,Harsh Critic


In [49]:
# STEP 33- Merge Archetypes Back Into Reviews
curated_reviews = curated_reviews.merge(
    persona_df[
        ["user_id", "archetype"]
    ],
    on="user_id",
    how="left"
)

curated_reviews.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,sentiment,archetype
0,rGI8UdsEGiGeETFiu1VK1w,WJnyWEe_YK7JO47fcovBVw,hRHhP3fhMy3LktPyQa3s_A,3,0,0,0,Good choice in Union Station St. Louis. I had...,2008-08-20 08:07:02,0.204545,Emotional Storyteller
1,Lun9ta1qn_pmuD1VqxiYmg,9gSuVhKyOx3Qn4oI6EQMaA,NwJoFxmYRDxVGXgPtrjQ3w,4,0,0,0,Went there for lunch and was pleasantly surpri...,2009-10-14 21:16:08,0.106667,Emotional Storyteller
2,Wu99UIXo1jGJeu97KCJzsw,iBQKwkuDvAdTM5gLWHgZwg,8uF-bhJFgT4Tn6DTb27viA,4,0,0,0,I don't think there is anything in this place ...,2017-12-16 01:54:03,0.403333,Warm Optimist
3,bY-J5JBKI9m8fiFm4CwCFA,ZybKys6Kg37xX2LMfvcntg,x4XDkWR9fgP4TItqMr8A8A,5,0,0,0,"Delicious, fresh, and friendly volunteers. Yes...",2014-06-11 16:43:45,0.430556,Reactive Reviewer
4,xHwfbcnzpIKXbFvHG8kRTQ,3kvIOBG06_rikfpk-EHIlQ,bXjnfT69E8DJinX-ifOofA,1,31,5,5,I've never had to write a review based on horr...,2012-11-07 17:45:30,0.131566,Emotional Storyteller


In [50]:
# TEST VALUE SIGNAL EXTRACTION

sample_review = curated_reviews.iloc[7]["text"]

print(sample_review[:500])

extract_value_signals(
    sample_review,
    value_taxonomy
)

Big fan of this place. They have the best sugar cookies and butter cake. (Although the butter cake starts getting stale within hours so eat it fast) They are always so nice; gave a cookie to my daughter today while we waited in line. Most importantly, I just leaned they make breakfast sandwiches on Fridays, Saturdays and Sundays. They are excellent and come with western potatoes and a croissant. You can see one half of a breakfast sandwich in my picture. All of this for just $6.99. A very good v


{'affordability': 1,
 'durability': 0,
 'service_quality': 0,
 'social_proof': 0,
 'time_efficiency': 1,
 'ambience': 0,
 'food_quality': 1,
 'electronics': 0,
 'fashion': 0,
 'books_media': 0,
 'convenience': 1,
 'social_experience': 0,
 'luxury': 0,
 'positive_exaggeration': 2,
 'negative_exaggeration': 0,
 'uncertainty': 0}

IMPORTANT

This is interpretable behavioral reasoning. It enables us to explain why the system believes something.

In [51]:
# STEP 35 - APPLY VALUE SIGNAL EXTRACTION
# Now we apply this function to all reviews to extract value signals for each review, 
# which will allow us to analyze the presence of different value-related themes in the 
# reviews and understand customer preferences.

from tqdm import tqdm

tqdm.pandas()

curated_reviews["value_signals"] = (
    curated_reviews["text"]
    .progress_apply(
        lambda x: extract_value_signals(
            x,
            value_taxonomy
        )
    )
)

  0%|          | 10/7413 [00:00<03:23, 36.38it/s]

100%|██████████| 7413/7413 [00:37<00:00, 197.42it/s]


In [52]:
# STEP 36 — EXPAND VALUE SIGNALS INTO COLUMNS
# This will allow us to analyze the presence of different value-related themes in the reviews and 
# understand customer preferences in a more structured way, enabling us to identify patterns and 
# insights related to the values expressed in the reviews.

value_df = pd.json_normalize(
    curated_reviews["value_signals"]
)

value_df.head(12)

,affordability,durability,service_quality,social_proof,time_efficiency,ambience,food_quality,electronics,fashion,books_media,convenience,social_experience,luxury,positive_exaggeration,negative_exaggeration,uncertainty
0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,2,0,0,0,8,0,0,0,0,0,0,0,0,0
4,0,0,3,0,1,0,0,0,0,0,0,2,0,0,0,2
5,1,1,1,0,2,0,5,0,0,0,0,0,0,1,0,0
6,0,0,1,0,0,1,1,0,0,0,0,0,0,1,0,0
7,1,0,0,0,1,0,1,0,0,0,1,0,0,2,0,0
8,1,0,0,0,0,0,0,0,0,0,1,0,0,3,0,0
9,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0


In [53]:
# STEP 37 — MERGE VALUE FEATURES
# Now each review contains inferred human values.
# This will allow us to analyze the presence of different value-related themes in the reviews 
# and understand customer preferences in a more structured way, enabling us to identify patterns 
# and insights related to the values expressed in the reviews.

curated_reviews = pd.concat(
    [curated_reviews, value_df],
    axis=1
)

curated_reviews.head()



,review_id,user_id,business_id,stars,useful,funny,cool,text,date,sentiment,...,food_quality,electronics,fashion,books_media,convenience,social_experience,luxury,positive_exaggeration,negative_exaggeration,uncertainty
0,rGI8UdsEGiGeETFiu1VK1w,WJnyWEe_YK7JO47fcovBVw,hRHhP3fhMy3LktPyQa3s_A,3,0,0,0,Good choice in Union Station St. Louis. I had...,2008-08-20 08:07:02,0.204545,...,0,0,0,0,0,0,0,0,0,0
1,Lun9ta1qn_pmuD1VqxiYmg,9gSuVhKyOx3Qn4oI6EQMaA,NwJoFxmYRDxVGXgPtrjQ3w,4,0,0,0,Went there for lunch and was pleasantly surpri...,2009-10-14 21:16:08,0.106667,...,2,0,0,0,0,0,0,0,0,0
2,Wu99UIXo1jGJeu97KCJzsw,iBQKwkuDvAdTM5gLWHgZwg,8uF-bhJFgT4Tn6DTb27viA,4,0,0,0,I don't think there is anything in this place ...,2017-12-16 01:54:03,0.403333,...,0,0,0,0,0,0,0,0,0,0
3,bY-J5JBKI9m8fiFm4CwCFA,ZybKys6Kg37xX2LMfvcntg,x4XDkWR9fgP4TItqMr8A8A,5,0,0,0,"Delicious, fresh, and friendly volunteers. Yes...",2014-06-11 16:43:45,0.430556,...,8,0,0,0,0,0,0,0,0,0
4,xHwfbcnzpIKXbFvHG8kRTQ,3kvIOBG06_rikfpk-EHIlQ,bXjnfT69E8DJinX-ifOofA,1,31,5,5,I've never had to write a review based on horr...,2012-11-07 17:45:30,0.131566,...,0,0,0,0,0,2,0,0,0,2


In [54]:
# STEP 38 — AGGREGATE VALUES PER USER
# Now we infer what users fundamentally care about.
# By aggregating the value signals at the user level, we can identify overarching themes and preferences 
# that characterize each user's reviews, providing deeper insights into their values and priorities when it 
# comes to restaurant experiences.

user_values = (
    curated_reviews
    .groupby("user_id")
    [
        list(value_taxonomy.keys())
    ]
    .mean()
    .reset_index()
)

user_values.head()

,user_id,affordability,durability,service_quality,social_proof,time_efficiency,ambience,food_quality,electronics,fashion,books_media,convenience,social_experience,luxury,positive_exaggeration,negative_exaggeration,uncertainty
0,-EX1hrPRBqNkVavtMllTCA,0.166667,0.222222,0.694444,0.000000,0.222222,0.083333,0.722222,0.000000,0.000000,0.0,0.250000,0.027778,0.000000,0.166667,0.027778,0.111111
1,-M7fUg7FrdGctKr5f_eMUQ,0.045455,0.136364,0.636364,0.000000,0.227273,0.500000,0.636364,0.000000,0.000000,0.0,0.090909,0.136364,0.045455,0.318182,0.045455,0.045455
2,-WM58wLjtlHlR91xVfM1FQ,0.000000,0.291667,0.416667,0.041667,0.250000,0.083333,1.166667,0.083333,0.041667,0.0,0.125000,0.333333,0.000000,0.666667,0.000000,0.125000
3,-qTtg1D3RidRa4cTB-ftwg,0.000000,0.133333,0.200000,0.000000,0.066667,0.000000,0.200000,0.000000,0.000000,0.0,0.333333,0.133333,0.000000,0.400000,0.000000,0.000000
4,02H49g16MdRoZKoX6IEoFA,0.136364,0.045455,1.136364,0.000000,0.136364,0.136364,0.636364,0.000000,0.090909,0.0,1.318182,0.090909,0.000000,0.272727,0.045455,0.045455


In [55]:
# STEP 39 — MERGE VALUES INTO PERSONAS
# This will allow us to enrich our user personas with insights about the values that are most important to them, 
# which can inform targeted marketing strategies, personalized recommendations, and a deeper understanding of customer segments.

persona_df = persona_df.merge(
    user_values,
    on="user_id",
    how="left"
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile,...,food_quality,electronics,fashion,books_media,convenience,social_experience,luxury,positive_exaggeration,negative_exaggeration,uncertainty
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic,"{'positivity': 'low', 'verbosity': 'high', 'em...",...,0.722222,0.000000,0.000000,0.0,0.250000,0.027778,0.000000,0.166667,0.027778,0.111111
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,0.636364,0.000000,0.000000,0.0,0.090909,0.136364,0.045455,0.318182,0.045455,0.045455
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,1.166667,0.083333,0.041667,0.0,0.125000,0.333333,0.000000,0.666667,0.000000,0.125000
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist,"{'positivity': 'high', 'verbosity': 'moderate'...",...,0.200000,0.000000,0.000000,0.0,0.333333,0.133333,0.000000,0.400000,0.000000,0.000000
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer,"{'positivity': 'mixed', 'verbosity': 'moderate...",...,0.636364,0.000000,0.090909,0.0,1.318182,0.090909,0.000000,0.272727,0.045455,0.045455


In [56]:
# STEP 40 — IDENTIFY DOMINANT VALUES
# This will help us understand the key values that define each user persona, allowing us to tailor our marketing strategies and recommendations to align with what matters most to each segment of our customer base.
# We identify the dominant value for each user by finding the value category with the highest average score in their reviews, which can provide insights into their core preferences and priorities.

value_columns = list(value_taxonomy.keys())

persona_df["dominant_value"] = (
    persona_df[value_columns]
    .idxmax(axis=1)
)

persona_df[
    ["user_id", "archetype", "dominant_value"]
].head(11)

,user_id,archetype,dominant_value
0,-EX1hrPRBqNkVavtMllTCA,Harsh Critic,food_quality
1,-M7fUg7FrdGctKr5f_eMUQ,Emotional Storyteller,service_quality
2,-WM58wLjtlHlR91xVfM1FQ,Emotional Storyteller,food_quality
3,-qTtg1D3RidRa4cTB-ftwg,Warm Optimist,positive_exaggeration
4,02H49g16MdRoZKoX6IEoFA,Reactive Reviewer,convenience
5,0650daOKAuufqymyOBe3cA,Warm Optimist,service_quality
6,06Yz-YYYa1U9PN37b6UniA,Emotional Storyteller,food_quality
7,0Tsu6-uhw_w9Z3W0Xlnazg,Harsh Critic,affordability
8,0xJwTzZuWOac7ufeTioeig,Emotional Storyteller,food_quality
9,175DuOm7IPiEy5f1KG9esA,Harsh Critic,food_quality


Personas are no longer generic reviewers. They now contain:

- emotional behavior
- archetypes
- behavioral style
- human priorities
- dominant values

PHASE 2 — TEMPORAL & EMOTIONAL DRIFT MODELING

This is where the system learns that humans evolve.

We will model:

Behavior	              Meaning
- rating drift	          becoming harsher/more generous over time
- sentiment drift	      emotional evolution
- preference drift	      changing tastes
- temporal behavior	      seasonality
- identity evolution	  life-stage transitions

In [1]:
# Example mapping – replace with your actual mapping from the notebook
archetype_to_dominant = {
    "Harsh Critic": "service_quality",
    "Emotional Storyteller": "ambience",
    "Social Butterfly": "social_proof",
    "Value Seeker": "affordability",
    "Quality Connoisseur": "food_quality"
    # ... add all your archetypes
}

In [2]:
import pandas as pd

persona_df = pd.read_csv("../outputs/persona_dataset.csv")
persona_df["dominant_value"] = persona_df["archetype"].map(archetype_to_dominant)

# Check for any missing (should be none)
print(persona_df["dominant_value"].isnull().sum())

# Save updated file
persona_df.to_csv("../outputs/persona_dataset.csv", index=False)

153


In [3]:
import pandas as pd
persona_df = pd.read_csv("../outputs/persona_dataset.csv")
print(persona_df.columns.tolist())
print(persona_df[['archetype', 'dominant_value']].head())

['user_id', 'avg_rating', 'rating_variance', 'review_count', 'avg_review_length', 'avg_sentiment', 'sentiment_variance', 'cluster', 'archetype', 'dominant_value']
               archetype   dominant_value
0           Harsh Critic  service_quality
1  Emotional Storyteller         ambience
2  Emotional Storyteller         ambience
3          Warm Optimist              NaN
4      Reactive Reviewer              NaN


In [4]:
# Load files
persona_df = pd.read_csv("../outputs/persona_dataset.csv")
unified_df = pd.read_csv("../outputs/unified_behavior_dataset.csv")

# Merge on user_id – keep all reviews, add archetype, dominant_value, cluster, avg_rating
unified_enriched = unified_df.merge(
    persona_df[['user_id', 'archetype', 'dominant_value', 'cluster', 'avg_rating']],
    on='user_id',
    how='left'
)

# Check missing archetypes (users not in persona_df will have NaN)
print(f"Rows without archetype: {unified_enriched['archetype'].isnull().sum()}")

# Save to new file
unified_enriched.to_csv("../outputs/unified_behavior_with_archetype.csv", index=False)
print("Saved unified_behavior_with_archetype.csv")

Rows without archetype: 28291
Saved unified_behavior_with_archetype.csv


In [5]:
df = pd.read_csv("../outputs/unified_behavior_with_archetype.csv")
print(df.columns.tolist())
print(df[['user_id', 'archetype', 'dominant_value']].head())

['domain', 'user_id', 'item_id', 'review_text', 'rating', 'timestamp', 'archetype', 'dominant_value', 'cluster', 'avg_rating']
                  user_id archetype dominant_value
0  mh_-eMZ6K5RLWhZyISBhwA       NaN            NaN
1  OyoGAe7OKpv6SyGZT5g77Q       NaN            NaN
2  8g_iMtfSiwikVnbP2etR0A       NaN            NaN
3  _7bHUi9Uuf5__HHc_Q8guQ       NaN            NaN
4  bcjbaE6dDog4jkNY91ncLQ       NaN            NaN


In [6]:
import pandas as pd

persona_df = pd.read_csv("../outputs/persona_dataset.csv")
unified_df = pd.read_csv("../outputs/unified_behavior_dataset.csv")

persona_users = set(persona_df['user_id'])
unified_users = set(unified_df['user_id'])

print(f"Persona users: {len(persona_users)}")
print(f"Unified users: {len(unified_users)}")
print(f"Overlap: {len(persona_users & unified_users)}")

Persona users: 300
Unified users: 18773
Overlap: 3


In [57]:
# STEP 41 — PREPARE TEMPORAL DATA
# We first ensure dates are proper datetime objects.

# Convert review dates to datetime

curated_reviews["date"] = pd.to_datetime(
    curated_reviews["date"]
)

print("Date conversion complete")

Date conversion complete


In [58]:
# STEP 42 — SORT USER REVIEWS CHRONOLOGICALLY


curated_reviews = curated_reviews.sort_values(
    by=["user_id", "date"]
)

curated_reviews.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,sentiment,...,food_quality,electronics,fashion,books_media,convenience,social_experience,luxury,positive_exaggeration,negative_exaggeration,uncertainty
985,6Vf1MkxDPrTcnuKYUldjFw,-EX1hrPRBqNkVavtMllTCA,agK5cXwnBQozM2M-5kLvzw,1,8,2,2,"We had food from this ""restaurant"" delivered t...",2011-12-15 12:02:27,-0.300000,...,2,0,0,0,0,0,0,0,0,0
7114,Aninptga9OOZsmi5gFNAjQ,-EX1hrPRBqNkVavtMllTCA,XnVmNQdmyhdCC90FRY9ZJQ,5,0,0,0,Had our Christmas party here this year. The s...,2011-12-16 23:14:10,0.540179,...,0,0,0,0,0,0,0,1,0,0
107,EiPqGnP5SRapBaOkvZJAug,-EX1hrPRBqNkVavtMllTCA,GBTPC53ZrG1ZBY3DT8Mbcw,4,1,0,1,"Went here on a lark, basically. I had enough ...",2012-02-04 17:10:19,0.298437,...,1,0,0,0,0,0,0,0,0,0
100,pL3LWEwcqaTLaa7c4o860A,-EX1hrPRBqNkVavtMllTCA,-9yzQQ0d_rcOD2CzdTNO_Q,5,1,1,2,"This is one of the ""retro"" McDonald's and it's...",2012-02-07 18:27:15,0.276420,...,0,0,0,0,0,0,0,0,0,0
2331,46Z1SVg6VJygnN3O95cFsw,-EX1hrPRBqNkVavtMllTCA,FEFi0AmjHzgSceeLYW3Glw,2,0,0,0,This location is inconsistent. Sometimes it b...,2012-02-07 18:49:57,-0.150000,...,0,0,0,0,2,0,0,0,0,0


WHY THIS MATTERS

Human evolution only makes sense over time.

Now reviews become behavioral timelines, instead of isolated events.

In [59]:
# STEP 43 — CREATE TEMPORAL USER HISTORIES
# Here we are creating temporal user histories by aggregating the review dates, star ratings, and sentiment scores for each user into lists.
# This will allow us to analyze how user preferences and sentiments evolve over time, providing insights into user behavior and trends in reviews.

temporal_histories = (
    curated_reviews
    .groupby("user_id")
    .agg({
        "date": list,
        "stars": list,
        "sentiment": list
    })
    .reset_index()
)

temporal_histories.head()

,user_id,date,stars,sentiment
0,-EX1hrPRBqNkVavtMllTCA,"[2011-12-15 12:02:27, 2011-12-16 23:14:10, 201...","[1, 5, 4, 5, 2, 5, 2, 5, 4, 1, 4, 2, 2, 5, 2, ...","[-0.3, 0.5401785714285714, 0.29843749999999997..."
1,-M7fUg7FrdGctKr5f_eMUQ,"[2014-01-03 07:44:42, 2014-01-03 07:56:13, 201...","[2, 4, 4, 1, 5, 4, 5, 5, 5, 5, 5, 4, 4, 5, 5, ...","[0.2, 0.3666666666666667, 0.054166666666666696..."
2,-WM58wLjtlHlR91xVfM1FQ,"[2014-04-21 22:33:53, 2014-08-31 18:31:10, 201...","[1, 5, 5, 5, 4, 5, 3, 5, 5, 5, 5, 5, 5, 5, 3, ...","[-0.13816183816183814, 0.33333333333333337, 0...."
3,-qTtg1D3RidRa4cTB-ftwg,"[2016-11-25 02:54:00, 2016-12-04 18:49:50, 201...","[5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 3, 5, 5, 5, 5]","[0.6666666666666666, 0.35, 0.3511904761904762,..."
4,02H49g16MdRoZKoX6IEoFA,"[2014-06-20 07:55:29, 2014-06-20 08:31:14, 201...","[4, 5, 4, 3, 5, 4, 5, 5, 5, 4, 5, 4, 5, 5, 5, ...","[0.014625000000000003, 0.3240165631469979, 0.3..."


WHAT THIS REPRESENTS

Each user now has:

chronological ratings
emotional trajectory
behavioral timeline

This represents a temporal identity.

In [60]:
temporal_histories["rating_drift"] = (
    temporal_histories["stars"]
    .apply(compute_rating_drift)
)

temporal_histories.head()

,user_id,date,stars,sentiment,rating_drift
0,-EX1hrPRBqNkVavtMllTCA,"[2011-12-15 12:02:27, 2011-12-16 23:14:10, 201...","[1, 5, 4, 5, 2, 5, 2, 5, 4, 1, 4, 2, 2, 5, 2, ...","[-0.3, 0.5401785714285714, 0.29843749999999997...",1
1,-M7fUg7FrdGctKr5f_eMUQ,"[2014-01-03 07:44:42, 2014-01-03 07:56:13, 201...","[2, 4, 4, 1, 5, 4, 5, 5, 5, 5, 5, 4, 4, 5, 5, ...","[0.2, 0.3666666666666667, 0.054166666666666696...",3
2,-WM58wLjtlHlR91xVfM1FQ,"[2014-04-21 22:33:53, 2014-08-31 18:31:10, 201...","[1, 5, 5, 5, 4, 5, 3, 5, 5, 5, 5, 5, 5, 5, 3, ...","[-0.13816183816183814, 0.33333333333333337, 0....",2
3,-qTtg1D3RidRa4cTB-ftwg,"[2016-11-25 02:54:00, 2016-12-04 18:49:50, 201...","[5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 3, 5, 5, 5, 5]","[0.6666666666666666, 0.35, 0.3511904761904762,...",0
4,02H49g16MdRoZKoX6IEoFA,"[2014-06-20 07:55:29, 2014-06-20 08:31:14, 201...","[4, 5, 4, 3, 5, 4, 5, 5, 5, 4, 5, 4, 5, 5, 5, ...","[0.014625000000000003, 0.3240165631469979, 0.3...",1


In [61]:
temporal_histories["sentiment_drift"] = (
    temporal_histories["sentiment"]
    .apply(compute_sentiment_drift)
)

temporal_histories.head()

,user_id,date,stars,sentiment,rating_drift,sentiment_drift
0,-EX1hrPRBqNkVavtMllTCA,"[2011-12-15 12:02:27, 2011-12-16 23:14:10, 201...","[1, 5, 4, 5, 2, 5, 2, 5, 4, 1, 4, 2, 2, 5, 2, ...","[-0.3, 0.5401785714285714, 0.29843749999999997...",1,0.365714
1,-M7fUg7FrdGctKr5f_eMUQ,"[2014-01-03 07:44:42, 2014-01-03 07:56:13, 201...","[2, 4, 4, 1, 5, 4, 5, 5, 5, 5, 5, 4, 4, 5, 5, ...","[0.2, 0.3666666666666667, 0.054166666666666696...",3,-0.309773
2,-WM58wLjtlHlR91xVfM1FQ,"[2014-04-21 22:33:53, 2014-08-31 18:31:10, 201...","[1, 5, 5, 5, 4, 5, 3, 5, 5, 5, 5, 5, 5, 5, 3, ...","[-0.13816183816183814, 0.33333333333333337, 0....",2,0.471495
3,-qTtg1D3RidRa4cTB-ftwg,"[2016-11-25 02:54:00, 2016-12-04 18:49:50, 201...","[5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 3, 5, 5, 5, 5]","[0.6666666666666666, 0.35, 0.3511904761904762,...",0,-0.263095
4,02H49g16MdRoZKoX6IEoFA,"[2014-06-20 07:55:29, 2014-06-20 08:31:14, 201...","[4, 5, 4, 3, 5, 4, 5, 5, 5, 4, 5, 4, 5, 5, 5, ...","[0.014625000000000003, 0.3240165631469979, 0.3...",1,0.270200


WHAT THIS MEANS

We are now detecting emotional evolution.

Examples:
- increasing positivity
- frustration accumulation
- declining enthusiasm
- emotional fatigue

In [62]:
temporal_histories["emotional_trajectory"] = (
    temporal_histories["sentiment_drift"]
    .apply(classify_drift)
)

temporal_histories.head(10)

,user_id,date,stars,sentiment,rating_drift,sentiment_drift,emotional_trajectory
0,-EX1hrPRBqNkVavtMllTCA,"[2011-12-15 12:02:27, 2011-12-16 23:14:10, 201...","[1, 5, 4, 5, 2, 5, 2, 5, 4, 1, 4, 2, 2, 5, 2, ...","[-0.3, 0.5401785714285714, 0.29843749999999997...",1,0.365714,emotionally_stable
1,-M7fUg7FrdGctKr5f_eMUQ,"[2014-01-03 07:44:42, 2014-01-03 07:56:13, 201...","[2, 4, 4, 1, 5, 4, 5, 5, 5, 5, 5, 4, 4, 5, 5, ...","[0.2, 0.3666666666666667, 0.054166666666666696...",3,-0.309773,emotionally_stable
2,-WM58wLjtlHlR91xVfM1FQ,"[2014-04-21 22:33:53, 2014-08-31 18:31:10, 201...","[1, 5, 5, 5, 4, 5, 3, 5, 5, 5, 5, 5, 5, 5, 3, ...","[-0.13816183816183814, 0.33333333333333337, 0....",2,0.471495,emotionally_stable
3,-qTtg1D3RidRa4cTB-ftwg,"[2016-11-25 02:54:00, 2016-12-04 18:49:50, 201...","[5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 3, 5, 5, 5, 5]","[0.6666666666666666, 0.35, 0.3511904761904762,...",0,-0.263095,emotionally_stable
4,02H49g16MdRoZKoX6IEoFA,"[2014-06-20 07:55:29, 2014-06-20 08:31:14, 201...","[4, 5, 4, 3, 5, 4, 5, 5, 5, 4, 5, 4, 5, 5, 5, ...","[0.014625000000000003, 0.3240165631469979, 0.3...",1,0.270200,emotionally_stable
5,0650daOKAuufqymyOBe3cA,"[2014-10-29 14:37:48, 2014-11-11 22:43:58, 201...","[5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5]","[0.18890977443609022, 0.11666666666666665, 0.5...",0,0.498590,emotionally_stable
6,06Yz-YYYa1U9PN37b6UniA,"[2015-11-11 02:59:40, 2015-11-11 03:09:13, 201...","[4, 3, 1, 5, 4, 5, 5, 5, 3, 4, 5, 5, 3, 4, 5, 4]","[0.6769999999999999, 0.19825757575757574, 0.01...",0,-0.512000,becoming_more_negative
7,0Tsu6-uhw_w9Z3W0Xlnazg,"[2015-11-09 16:59:35, 2015-11-09 17:05:24, 201...","[1, 2, 5, 1, 1, 3, 3, 1, 2, 1, 2, 1, 1, 1, 1, 1]","[0.02291666666666667, 0.40208333333333335, 0.5...",0,0.056944,emotionally_stable
8,0xJwTzZuWOac7ufeTioeig,"[2012-10-30 21:59:15, 2013-03-02 18:33:18, 201...","[5, 5, 4, 4, 4, 1, 5, 5, 5, 3, 5, 4, 2, 5, 4, ...","[0.5666666666666667, 0.26583092833092836, 0.25...",-4,-0.829167,becoming_more_negative
9,175DuOm7IPiEy5f1KG9esA,"[2016-05-14 02:41:10, 2016-06-01 01:14:13, 201...","[4, 5, 1, 5, 5, 5, 5, 3, 5, 1, 4, 5, 3, 2, 5, ...","[0.13571428571428573, 0.06441197691197692, 0.0...",0,0.071104,emotionally_stable


In [63]:
# STEP 47 — SUMMARIZE EMOTIONAL TRAJECTORIES
# This will help us understand the distribution of emotional trajectories among users, 
# providing insights into how user sentiments evolve over time and whether there are common patterns in emotional changes.

stable = (temporal_histories["emotional_trajectory"] == "emotionally_stable").sum()
positive = (temporal_histories["emotional_trajectory"] == "becoming_more_positive").sum()
negative = (temporal_histories["emotional_trajectory"] == "becoming_more_negative").sum()

print("emotionally_stable:", stable)
print("becoming_more_positive:", positive)
print("becoming_more_negative:", negative)

emotionally_stable: 271
becoming_more_positive: 11
becoming_more_negative: 18


In [64]:
# STEP 48 — INSPECT USERS WITH EMOTIONAL DRIFT
# This will allow us to qualitatively analyze the reviews of users who exhibit significant emotional drift, 
# providing insights into the factors that may contribute to changes in user sentiment over time and how these changes manifest in their reviews.

for trajectory in [
    "becoming_more_positive",
    "becoming_more_negative"
]:

    print("\n" + "="*80)
    print(f"TRAJECTORY: {trajectory}")
    print("="*80)

    # Get sample users
    sampled_users = (
        temporal_histories[
            temporal_histories["emotional_trajectory"]
            == trajectory
        ]
        .sample(3, random_state=42)
    )

    for _, user_row in sampled_users.iterrows():

        user_id = user_row["user_id"]

        print("\n" + "#"*70)
        print(f"USER ID: {user_id}")
        print(f"Trajectory: {trajectory}")
        print(f"Rating Drift: {user_row['rating_drift']}")
        print(f"Sentiment Drift: {user_row['sentiment_drift']}")
        print("#"*70)

        # Get chronological reviews
        user_reviews = (
            curated_reviews[
                curated_reviews["user_id"] == user_id
            ]
            .sort_values("date")
        )

        # FIRST REVIEW
        first_review = user_reviews.iloc[0]

        print("\nFIRST REVIEW")
        print("-"*50)
        print(f"Date: {first_review['date']}")
        print(f"Stars: {first_review['stars']}")
        print(f"Sentiment: {first_review['sentiment']}")

        print(first_review["text"][:500])

        # LAST REVIEW
        last_review = user_reviews.iloc[-1]

        print("\nLAST REVIEW")
        print("-"*50)
        print(f"Date: {last_review['date']}")
        print(f"Stars: {last_review['stars']}")
        print(f"Sentiment: {last_review['sentiment']}")

        print(last_review["text"][:500])

        print("\n\n")


TRAJECTORY: becoming_more_positive

######################################################################
USER ID: OGxTI5DMFWrPWp65_wHWGQ
Trajectory: becoming_more_positive
Rating Drift: 4
Sentiment Drift: 0.7169934640522876
######################################################################

FIRST REVIEW
--------------------------------------------------
Date: 2013-11-10 22:06:47
Stars: 1
Sentiment: 0.060784313725490244
Attention Business Owner,
I first tried your restaurant, Cheba Hut in Tucson, about a month ago and thought it was fun, quirky, quick and tasty. I liked it enough to recommend it to others. Which is why it was stunning to have the complete opposite experience today. Bad customer service takes some of the good taste from food. The general manager, who says he is your brother provided by far the worst CS experience I have had in a while (talking over me, correcting me, etc). Even the employee we w

LAST REVIEW
--------------------------------------------------
Date:

In [65]:
# STEP 49 — MERGE TEMPORAL SIGNALS INTO PERSONAS
# This will allow us to enrich our user personas with insights about how their sentiments and ratings evolve over time,

persona_df = persona_df.merge(
    temporal_histories[
        [
            "user_id",
            "rating_drift",
            "sentiment_drift",
            "emotional_trajectory"
        ]
    ],
    on="user_id",
    how="left"
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile,...,convenience,social_experience,luxury,positive_exaggeration,negative_exaggeration,uncertainty,dominant_value,rating_drift,sentiment_drift,emotional_trajectory
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic,"{'positivity': 'low', 'verbosity': 'high', 'em...",...,0.250000,0.027778,0.000000,0.166667,0.027778,0.111111,food_quality,1,0.365714,emotionally_stable
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,0.090909,0.136364,0.045455,0.318182,0.045455,0.045455,service_quality,3,-0.309773,emotionally_stable
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,0.125000,0.333333,0.000000,0.666667,0.000000,0.125000,food_quality,2,0.471495,emotionally_stable
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist,"{'positivity': 'high', 'verbosity': 'moderate'...",...,0.333333,0.133333,0.000000,0.400000,0.000000,0.000000,positive_exaggeration,0,-0.263095,emotionally_stable
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer,"{'positivity': 'mixed', 'verbosity': 'moderate...",...,1.318182,0.090909,0.000000,0.272727,0.045455,0.045455,convenience,1,0.270200,emotionally_stable


WHAT WE NOW POSSESS: A real cognitive persona engine.

The personas now include:

Dimension	         Meaning
archetype	         behavioral identity
dominant_value	     priorities
sentiment	         emotionality
drift	             evolution
trajectory	         temporal behavior




PHASE 3: PREFERENCE DRIFT MODELLING

This models changing interests over time.

Example:

2019:
cheap fast food

2023:
upscale aesthetic dining

That implies income change, identity change, maturity, lifestyle evolution etc.

We will detect:

- changing restaurant categories
- changing review topics
- evolving priorities
- shifting preferences

In [66]:
# STEP 50 — EXTRACT BUSINESS CATEGORIES
# 

business_subset = business_df[
    ["business_id", "categories"]
].copy()

business_subset.head()

,business_id,categories
0,Pns2l4eNsfO8kk83dixA6A,"Doctors, Traditional Chinese Medicine, Naturop..."
1,mpf3x-BjTdTEA3yCZrAYPw,"Shipping Centers, Local Services, Notaries, Ma..."
2,tUFrWirKiKi_TAnsVWINQQ,"Department Stores, Shopping, Fashion, Home & G..."
3,MTSW4McQd7CbVtyjqoe9mw,"Restaurants, Food, Bubble Tea, Coffee & Tea, B..."
4,mWMc6_wTdE0EUBKIGXDVfA,"Brewpubs, Breweries, Food"


In [67]:
# STEP 51 — MERGE BUSINESS CATEGORIES INTO REVIEWS

curated_reviews = curated_reviews.merge(
    business_subset,
    on="business_id",
    how="left"
)

curated_reviews.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,sentiment,...,electronics,fashion,books_media,convenience,social_experience,luxury,positive_exaggeration,negative_exaggeration,uncertainty,categories
0,6Vf1MkxDPrTcnuKYUldjFw,-EX1hrPRBqNkVavtMllTCA,agK5cXwnBQozM2M-5kLvzw,1,8,2,2,"We had food from this ""restaurant"" delivered t...",2011-12-15 12:02:27,-0.300000,...,0,0,0,0,0,0,0,0,0,"Italian, American (Traditional), Restaurants, ..."
1,Aninptga9OOZsmi5gFNAjQ,-EX1hrPRBqNkVavtMllTCA,XnVmNQdmyhdCC90FRY9ZJQ,5,0,0,0,Had our Christmas party here this year. The s...,2011-12-16 23:14:10,0.540179,...,0,0,0,0,0,0,1,0,0,"Restaurants, Steakhouses, Seafood, Nightlife, ..."
2,EiPqGnP5SRapBaOkvZJAug,-EX1hrPRBqNkVavtMllTCA,GBTPC53ZrG1ZBY3DT8Mbcw,4,1,0,1,"Went here on a lark, basically. I had enough ...",2012-02-04 17:10:19,0.298437,...,0,0,0,0,0,0,0,0,0,"German, Restaurants, Seafood, Cocktail Bars, F..."
3,pL3LWEwcqaTLaa7c4o860A,-EX1hrPRBqNkVavtMllTCA,-9yzQQ0d_rcOD2CzdTNO_Q,5,1,1,2,"This is one of the ""retro"" McDonald's and it's...",2012-02-07 18:27:15,0.276420,...,0,0,0,0,0,0,0,0,0,"Fast Food, Restaurants, Coffee & Tea, Food, Bu..."
4,46Z1SVg6VJygnN3O95cFsw,-EX1hrPRBqNkVavtMllTCA,FEFi0AmjHzgSceeLYW3Glw,2,0,0,0,This location is inconsistent. Sometimes it b...,2012-02-07 18:49:57,-0.150000,...,0,0,0,2,0,0,0,0,0,"Food, Burgers, Coffee & Tea, Fast Food, Restau..."


In [68]:
# STEP 52 — CLEAN CATEGORY TEXT

curated_reviews["categories"] = (
    curated_reviews["categories"]
    .fillna("")
)

In [69]:
# STEP 53 — CREATE SIMPLE CUISINE TAGS
# We now infer preference domains.


cuisine_keywords = [
    "Mexican",
    "Italian",
    "Chinese",
    "Japanese",
    "Thai",
    "Indian",
    "American",
    "Mediterranean",
    "Korean",
    "French",
    "Pizza",
    "Seafood",
    "Burgers",
    "Cafe",
    "Bars"
]

In [70]:
# STEP 54 — EXTRACT CUISINE PREFERENCES

def extract_cuisines(category_text):

    found = []

    for cuisine in cuisine_keywords:

        if cuisine.lower() in category_text.lower():
            found.append(cuisine)

    return found

In [71]:
curated_reviews["cuisines"] = (
    curated_reviews["categories"]
    .apply(extract_cuisines)
)

curated_reviews[
    ["categories", "cuisines"]
].head()

,categories,cuisines
0,"Italian, American (Traditional), Restaurants, ...","[Italian, American]"
1,"Restaurants, Steakhouses, Seafood, Nightlife, ...","[American, Seafood, Bars]"
2,"German, Restaurants, Seafood, Cocktail Bars, F...","[American, French, Seafood, Bars]"
3,"Fast Food, Restaurants, Coffee & Tea, Food, Bu...",[Burgers]
4,"Food, Burgers, Coffee & Tea, Fast Food, Restau...",[Burgers]


In [72]:
# STEP 55 — SPLIT EARLY VS RECENT BEHAVIOR
# We compare past self vs current self.

def split_temporal_preferences(user_df):

    user_df = user_df.sort_values("date")

    midpoint = len(user_df) // 2

    early = user_df.iloc[:midpoint]
    recent = user_df.iloc[midpoint:]

    return early, recent

In [73]:
# STEP 56: DETECT PREFERENCE DRIFT

from collections import Counter

preference_drift_results = []

for user_id, user_df in curated_reviews.groupby("user_id"):

    if len(user_df) < 6:
        continue

    early, recent = split_temporal_preferences(user_df)

    early_cuisines = [
        cuisine
        for sublist in early["cuisines"]
        for cuisine in sublist
    ]

    recent_cuisines = [
        cuisine
        for sublist in recent["cuisines"]
        for cuisine in sublist
    ]

    early_top = Counter(early_cuisines).most_common(3)
    recent_top = Counter(recent_cuisines).most_common(3)

    preference_drift_results.append({
        "user_id": user_id,
        "early_preferences": early_top,
        "recent_preferences": recent_top
    })

In [74]:
# STEP 57 — CREATE PREFERENCE DRIFT DATAFRAME

preference_drift_df = pd.DataFrame(
    preference_drift_results
)

preference_drift_df.head()

,user_id,early_preferences,recent_preferences
0,-EX1hrPRBqNkVavtMllTCA,"[(American, 7), (Burgers, 7), (Bars, 4)]","[(Burgers, 4), (American, 3), (Bars, 3)]"
1,-M7fUg7FrdGctKr5f_eMUQ,"[(Mexican, 5), (American, 4), (Seafood, 4)]","[(Bars, 7), (American, 5), (Mediterranean, 2)]"
2,-WM58wLjtlHlR91xVfM1FQ,"[(American, 3), (Mexican, 2), (Burgers, 2)]","[(Cafe, 2), (Mediterranean, 2), (Seafood, 2)]"
3,-qTtg1D3RidRa4cTB-ftwg,"[(American, 4), (Seafood, 3), (Cafe, 1)]","[(American, 7), (Bars, 6), (Cafe, 2)]"
4,02H49g16MdRoZKoX6IEoFA,"[(American, 4), (Bars, 2), (Pizza, 2)]","[(Mexican, 3), (Burgers, 2), (American, 2)]"


In [75]:
# Inspect a few users
preference_drift_df.sample(15)

,user_id,early_preferences,recent_preferences
17,21u_7llMMrgnWD9nyWbgAA,"[(Bars, 7), (American, 5), (Italian, 3)]","[(Italian, 8), (Pizza, 6), (Bars, 4)]"
197,fr7ac23SbYpBNl-Z-nyv3Q,"[(American, 7), (Bars, 6), (Mexican, 6)]","[(American, 10), (Mexican, 3), (Burgers, 2)]"
88,IyLBcbpdOP-MjBeDSzdjEw,"[(Bars, 6), (American, 5), (Burgers, 4)]","[(Bars, 10), (Chinese, 7), (Korean, 3)]"
260,tGoHWnx270FjmoZssSqGyQ,"[(American, 4), (Cafe, 3), (Bars, 2)]","[(American, 4), (Bars, 3), (Italian, 1)]"
201,gjkeN_T14s0Y1tzQaheDPQ,"[(American, 5), (Bars, 4), (Pizza, 1)]","[(Bars, 5), (American, 4), (Italian, 2)]"
225,mD-IgInk0o8pXrJI-P8pYA,"[(American, 5), (Bars, 1)]","[(American, 4), (Burgers, 3), (Bars, 2)]"
247,q4Oz1c_OjGu_a8ZkP53Vnw,"[(Bars, 7), (Japanese, 6), (Mexican, 2)]","[(American, 12), (Bars, 10), (Seafood, 3)]"
198,gEhxkSS-_H8yS9K1cA7TQA,"[(Chinese, 3), (Indian, 2), (Italian, 1)]","[(American, 2), (Bars, 2), (Chinese, 2)]"
158,Z8Dv-N42EfoCkQ_MP6npmA,"[(Bars, 5), (American, 3), (Seafood, 1)]","[(Bars, 3), (American, 3), (Pizza, 2)]"
24,4jQKzMT5Fpjs2bUhPjVwHw,"[(American, 8), (Bars, 6), (Seafood, 3)]","[(Bars, 8), (American, 5), (Pizza, 3)]"


Possible interpretation for preference drift of these users:

maturing tastes, luxury orientation, lifestyle evolution

This shows human evolution modeling.

PHASE 4 — LINGUISTIC STYLE MODELING
This is critical for Task A review simulation. Because humans are not only what they value or how they feel. They are also how they speak.

In this phase, we are building an agent that understands varying styles and dimensions to review writing	
Example:
- storytelling (long narrative reviews)
- analytical	(structured critique)
- emotionality	(expressive language)
- sarcasm	(indirect criticism)
- slang	(casual speech)
- enthusiasm	(exaggerated positivity)
- formality	(polished vs casual)

Build:

- review style metrics
- punctuation behavior
- capitalization style
- emotional intensity
- storytelling tendency
- slang detection
- expressiveness

In [76]:
# STEP 58 — CREATE REVIEW LENGTH FEATURE
# This will allow us to analyze how the length of reviews correlates with user personas, sentiments, and preferences, providing insights into user engagement and expression in their reviews.

curated_reviews["review_length"] = (
    curated_reviews["text"]
    .apply(len)
)

In [77]:
# STEP 59 — EXCLAMATION USAGE
# This captures emotional expressiveness.
# By counting the number of exclamation marks in reviews, we can gain insights into the emotional intensity and 
# expressiveness of users, which may correlate with certain personas or sentiments in their reviews.

curated_reviews["exclamation_count"] = (
    curated_reviews["text"]
    .str.count("!")
)

In [78]:
# STEP 60 — QUESTION MARK USAGE
# This captures sarcasm, confusion, and conversational tone
# By counting the number of question marks in reviews, we can gain insights into the conversational tone in user reviews, 
# which may correlate with certain personas or sentiments.

curated_reviews["question_count"] = (
    curated_reviews["text"]
    .str.count(r"\?")
)

In [79]:
# STEP 61 — UPPERCASE EMPHASIS
# Humans use caps for excitemenT, anger, emphasis
# By calculating the ratio of uppercase letters in reviews, we can gain insights into the emotional intensity and 
# emphasis in user reviews, which may correlate with certain personas or sentiments.

def uppercase_ratio(text):

    if len(text) == 0:
        return 0

    uppercase_chars = sum(
        1 for c in text if c.isupper()
    )

    return uppercase_chars / len(text)

In [80]:
curated_reviews["uppercase_ratio"] = (
    curated_reviews["text"]
    .apply(uppercase_ratio)
)

In [81]:
# STEP 62 — STORYTELLING DETECTION

storytelling_keywords = {
    
    # Temporal sequencing (story moves through time)
    "time_sequence": [
        "first", "then", "next", "after that", "finally", "later",
        "before", "when", "while", "as soon as", "suddenly",
        "eventually", "in the end", "at first", "initially"
    ],
    
    # Personal experience framing
    "personal_anchor": [
        "I arrived", "I went", "I walked in", "I met", "I saw",
        "I decided", "I asked", "I told", "I thought", "I felt",
        "my friend and I", "we entered", "they welcomed us"
    ],
    
    # Scene setting (time, place, atmosphere)
    "scene_setting": [
        "it was a", "the weather", "the place was", "the atmosphere",
        "as soon as I entered", "the smell of", "the music was playing",
        "it was crowded", "quiet", "bustling", "dark", "bright"
    ],
    
    # Dialogue / quoted speech (strong storytelling signal)
    "dialogue": [
        "said", "asked", "told me", "shouted", "whispered",
        "called me", "replied", "answered", "he said '", "she said '",
        "I said '", "then he goes '", "I was like '"
    ],
    
    # Emotional arc (build‑up, climax, resolution)
    "emotion_arc": [
        "I was excited", "disappointed", "surprised", "shocked",
        "relieved", "angry", "happy", "sad", "confused",
        "my heart sank", "I couldn't believe", "I almost cried",
        "I laughed", "I regretted", "I was so happy that"
    ],
    
    # Conflict & resolution (classic story structure)
    "conflict_resolution": [
        "problem was", "issue came up", "something went wrong",
        "unfortunately", "luckily", "thankfully", "to make matters worse",
        "in the end", "they fixed it", "we sorted it out", "I complained"
    ],
    
    # Nigerian‑specific storytelling markers (colloquial narrative style)
    "naija_narrative": [
        "so I tell am", "the guy come say", "I just dey go",
        "immediately I enter", "see as e be", "before I know",
        "the next thing", "as I was coming", "I reach there",
        "the seller tell me", "my sister say make I try am",
        "story for another day", "I no fit shout", "you won't believe"
    ],
    
    # Exaggerated storytelling (typical of oral tradition)
    "hyperbole_narrative": [
        "I waited for years", "the longest hour of my life",
        "everybody in Lagos", "the whole market", "I almost died",
        "I swear down", "e be like film", "like a movie scene"
    ],
    
    # Reflective / moral ending (story with lesson)
    "lesson_ending": [
        "I learned that", "from that day", "never again will I",
        "that taught me", "the moral is", "I now know that",
        "if there's one thing I learned"
    ]
}

In [82]:
# STEP 63 — DETECT STORYTELLING STYLE
# By counting the presence of storytelling keywords in reviews, we can identify users who tend to write narrative-style reviews, which may correlate with certain personas or sentiments.
def storytelling_score(text):

    text = text.lower()

    score = 0

    for keyword in storytelling_keywords:

        score += text.count(keyword)

    return score

In [83]:
curated_reviews["storytelling_score"] = (
    curated_reviews["text"]
    .apply(storytelling_score)
)

In [84]:
# STEP 64 - SLANG & CASUAL LANGUAGE DETECTION

slang_words = [
    
]

slang_words = {

    "frequent": [
        "lol", "omg", "wtf", "super", "kinda", "totally", "literally", "crazy", "weird",
        "awesome", "it's giving"
    ],

    # Pidgin English staples (high‑frequency casual markers)
    "pidgin_staples": [
        "abeg", "na wa o", "wahala", "sef", "nko", "abi", "na", "o",
        "ooh", "sha", "kpa", "kwa", "nawa", "mtchew", "chei", "chai",
        "walahi", "biko", "ndo", "jare", "gan", "sabi"
    ],
    
    # Casual greetings & exclamations
    "exclamations": [
        "see finish", "see me see trouble", "oga", "madam", "boss",
        "my brother", "my sister", "my guy", "my dear", "babe",
        "sis", "bro", "chief", "alhaji", "mama", "papa"
    ],
    
    # Slang for good / bad (informal evaluation)
    "informal_quality": [
        "sweet", "smooth", "soft", "hard", "rough", "wahala",
        "bomb", "lit", "trash", "junk", "wayo", "419", "fake life",
        "original", "oloje", "gbege", "yanfu", "gbese", "sapa"
    ],
    
    # Everyday actions (casual verb forms)
    "casual_verbs": [
        "grab", "chop", "flash", "shayo", "logde", "comot",
        "waka", "manage", "hustle", "hammer", "ball", "settle",
        "run", "form", "fake", "throway", "carry go"
    ],
    
    # Fillers & discourse markers (spoken language features)
    "fillers": [
        "like", "you know", "I mean", "actually", "basically",
        "honestly", "literally", "so yeah", "anyway", "well",
        "the thing is", "see e be like", "make I talk true"
    ],
    
    # Abbreviations & shortenings (text‑speak)
    "abbreviations": [
        "pls", "cos", "bcuz", "u", "ur", "d", "dey", "dem",
        "wen", "den", "nao", "da", "dis", "dat", "dese", "dose",
        "wifi", "data", "app", "phone", "lappy", "btw", "idk", "imo"
    ],
    
    # Repetition for emphasis (casual intensification)
    "repetition": [
        "very very", "too too", "so so", "like like", "many many",
        "plenty plenty", "small small", "quick quick", "everywhere everywhere"
    ],
    
    # Sentence‑final particles (casual tone)
    "final_particles": [
        "o", "ooh", "sha", "na", "abi", "right?", "you get?",
        "you feel me?", "you see?", "ehn?", "not?", "so?"
    ],
    
    # Colloquial time references
    "casual_time": [
        "this morning", "yesterday night", "tomorrow morning",
        "next tomorrow", "today today", "immediately", "straight away",
        "quickly quick", "slowly slowly", "by force by force"
    ],
    
    # Money & transaction slang (very common in reviews)
    "money_slang": [
        "cash", "kudi", "money", "naira", "kobo", "bills", "change",
        "balance", "credit", "loan", "subscription", "airtime", "data"
    ]
}

In [85]:
# STEP 65 — COMPUTE SLANG SCORE

def slang_score(text):

    text = text.lower()

    score = 0

    for word in slang_words:

        score += text.count(word)

    return score

In [86]:
curated_reviews["slang_score"] = (
    curated_reviews["text"]
    .apply(slang_score)
)

In [87]:
# STEP 66 — AGGREGATE LINGUISTIC FEATURES PER USER
# Now we create linguistic identities.
# 

linguistic_features = (
    curated_reviews
    .groupby("user_id")
    [
        [
            "review_length",
            "exclamation_count",
            "question_count",
            "uppercase_ratio",
            "storytelling_score",
            "slang_score"
        ]
    ]
    .mean()
    .reset_index()
)

linguistic_features.head(8)

,user_id,review_length,exclamation_count,question_count,uppercase_ratio,storytelling_score,slang_score
0,-EX1hrPRBqNkVavtMllTCA,390.194444,0.472222,0.166667,0.044430,0.0,0.0
1,-M7fUg7FrdGctKr5f_eMUQ,358.727273,0.363636,0.090909,0.022010,0.0,0.0
2,-WM58wLjtlHlR91xVfM1FQ,632.666667,2.166667,0.375000,0.040376,0.0,0.0
3,-qTtg1D3RidRa4cTB-ftwg,195.666667,1.333333,0.200000,0.042992,0.0,0.0
4,02H49g16MdRoZKoX6IEoFA,406.545455,0.000000,0.000000,0.034632,0.0,0.0
5,0650daOKAuufqymyOBe3cA,255.937500,0.312500,0.000000,0.041413,0.0,0.0
6,06Yz-YYYa1U9PN37b6UniA,777.625000,0.187500,0.062500,0.030440,0.0,0.0
7,0Tsu6-uhw_w9Z3W0Xlnazg,411.125000,1.187500,0.750000,0.025743,0.0,0.0


In [88]:
# STEP 67 — MERGE INTO PERSONAS

persona_df = persona_df.merge(
    linguistic_features,
    on="user_id",
    how="left"
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile,...,dominant_value,rating_drift,sentiment_drift,emotional_trajectory,review_length,exclamation_count,question_count,uppercase_ratio,storytelling_score,slang_score
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic,"{'positivity': 'low', 'verbosity': 'high', 'em...",...,food_quality,1,0.365714,emotionally_stable,390.194444,0.472222,0.166667,0.044430,0.0,0.0
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,service_quality,3,-0.309773,emotionally_stable,358.727273,0.363636,0.090909,0.022010,0.0,0.0
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,food_quality,2,0.471495,emotionally_stable,632.666667,2.166667,0.375000,0.040376,0.0,0.0
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist,"{'positivity': 'high', 'verbosity': 'moderate'...",...,positive_exaggeration,0,-0.263095,emotionally_stable,195.666667,1.333333,0.200000,0.042992,0.0,0.0
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer,"{'positivity': 'mixed', 'verbosity': 'moderate...",...,convenience,1,0.270200,emotionally_stable,406.545455,0.000000,0.000000,0.034632,0.0,0.0


In [89]:
# STEP 68 — CREATE COMMUNICATION STYLES
# Now we interpret linguistic behavior.

def communication_style(row):

    if row["storytelling_score"] > 2:
        return "narrative"

    elif row["review_length"] > 1200:
        return "deeply_descriptive"

    elif row["slang_score"] > 1:
        return "casual_expressive"

    elif row["uppercase_ratio"] > 0.05:
        return "emotionally_emphatic"

    else:
        return "balanced"

In [90]:
 
persona_df["communication_style"] = (
    persona_df.apply(
        communication_style,
        axis=1
    )
)

persona_df[
    [
        "user_id",
        "archetype",
        "communication_style"
    ]
].head(20)

,user_id,archetype,communication_style
0,-EX1hrPRBqNkVavtMllTCA,Harsh Critic,balanced
1,-M7fUg7FrdGctKr5f_eMUQ,Emotional Storyteller,balanced
2,-WM58wLjtlHlR91xVfM1FQ,Emotional Storyteller,balanced
3,-qTtg1D3RidRa4cTB-ftwg,Warm Optimist,balanced
4,02H49g16MdRoZKoX6IEoFA,Reactive Reviewer,balanced
5,0650daOKAuufqymyOBe3cA,Warm Optimist,balanced
6,06Yz-YYYa1U9PN37b6UniA,Emotional Storyteller,balanced
7,0Tsu6-uhw_w9Z3W0Xlnazg,Harsh Critic,balanced
8,0xJwTzZuWOac7ufeTioeig,Emotional Storyteller,balanced
9,175DuOm7IPiEy5f1KG9esA,Harsh Critic,balanced


personas now contain:

- Dimension	(Meaning)
- archetype	(psychological identity)
- dominant_value	(motivations)
- drift	(evolution)
- communication_style	(speaking behavior)

In [91]:
# STEP 69: VIEW ALL AVAILABLE COMMUNICATION STYLES

persona_df["communication_style"].value_counts()

communication_style
balanced                281
deeply_descriptive       16
emotionally_emphatic      3
Name: count, dtype: int64

In [92]:
# STEP 70: GENERATE USERS BY STYLE
# Narrative Users
narrative_users = persona_df[
    persona_df["communication_style"] == "narrative"
]

narrative_users.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile,...,rating_drift,sentiment_drift,emotional_trajectory,review_length,exclamation_count,question_count,uppercase_ratio,storytelling_score,slang_score,communication_style


In [93]:
# Casual Expressive Users
casual_users = persona_df[
    persona_df["communication_style"] == "casual_expressive"
]

casual_users.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile,...,rating_drift,sentiment_drift,emotional_trajectory,review_length,exclamation_count,question_count,uppercase_ratio,storytelling_score,slang_score,communication_style


In [94]:
# Emotionally Emphatic Users
emphatic_users = persona_df[
    persona_df["communication_style"] == "emotionally_emphatic"
]

emphatic_users.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile,...,rating_drift,sentiment_drift,emotional_trajectory,review_length,exclamation_count,question_count,uppercase_ratio,storytelling_score,slang_score,communication_style
162,Zna994E0FpeKVhMEENtytg,3.562500,1.590335,16,213.125000,0.345808,0.278232,1,Reactive Reviewer,"{'positivity': 'mixed', 'verbosity': 'moderate...",...,-3,-0.221104,emotionally_stable,213.125000,0.562500,0.187500,0.056443,0.0,0.0,emotionally_emphatic
194,fLDeP8zz1UBsEypz3BTbJQ,4.382353,1.326072,34,306.235294,0.236101,0.197013,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,0,0.212398,emotionally_stable,306.235294,1.529412,0.058824,0.058160,0.0,0.0,emotionally_emphatic
215,k0p0RWMjvv8Ps1SUSuYL1A,3.850000,1.308877,20,272.600000,0.227015,0.204661,1,Reactive Reviewer,"{'positivity': 'mixed', 'verbosity': 'moderate...",...,0,0.164886,emotionally_stable,272.600000,0.200000,0.100000,0.051280,0.0,0.0,emotionally_emphatic


In [95]:
# Deeply Descriptive Users
descriptive_users = persona_df[
    persona_df["communication_style"] == "deeply_descriptive"
]

descriptive_users.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile,...,rating_drift,sentiment_drift,emotional_trajectory,review_length,exclamation_count,question_count,uppercase_ratio,storytelling_score,slang_score,communication_style
19,2xNYCJxZOrhVsrVMz8mlDQ,3.911765,0.933149,34,2181.441176,0.228747,0.066279,4,Deep Experience Analyst,"{'positivity': 'moderate', 'verbosity': 'very ...",...,1,0.063800,emotionally_stable,2181.441176,1.470588,0.882353,0.020633,0.0,0.000000,deeply_descriptive
38,9VdyNBdQbaZTrFDmD49e7A,3.736842,1.097578,19,2119.368421,0.181577,0.117300,4,Deep Experience Analyst,"{'positivity': 'moderate', 'verbosity': 'very ...",...,1,0.188542,emotionally_stable,2119.368421,1.789474,0.578947,0.017747,0.0,0.052632,deeply_descriptive
45,Bezh6tDmCrL0-SnsV6CneA,4.000000,0.617213,22,1221.181818,0.189392,0.105360,4,Deep Experience Analyst,"{'positivity': 'moderate', 'verbosity': 'very ...",...,-1,-0.188326,emotionally_stable,1221.181818,0.772727,0.136364,0.030555,0.0,0.000000,deeply_descriptive
54,Do8oscF3LjCl-_pXrmgLXA,4.041667,0.954585,24,2228.416667,0.178727,0.066896,4,Deep Experience Analyst,"{'positivity': 'moderate', 'verbosity': 'very ...",...,1,-0.062266,emotionally_stable,2228.416667,0.625000,0.375000,0.021523,0.0,0.000000,deeply_descriptive
64,FAglQiRDGCRaVsuMkwifQQ,3.777778,0.847319,27,1577.518519,0.197376,0.111403,4,Deep Experience Analyst,"{'positivity': 'moderate', 'verbosity': 'very ...",...,-1,0.209020,emotionally_stable,1577.518519,2.296296,0.148148,0.028980,0.0,0.037037,deeply_descriptive


In [96]:
# Balanced Users
balanced_users = persona_df[
    persona_df["communication_style"] == "balanced"
]

balanced_users.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile,...,rating_drift,sentiment_drift,emotional_trajectory,review_length,exclamation_count,question_count,uppercase_ratio,storytelling_score,slang_score,communication_style
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic,"{'positivity': 'low', 'verbosity': 'high', 'em...",...,1,0.365714,emotionally_stable,390.194444,0.472222,0.166667,0.044430,0.0,0.0,balanced
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,3,-0.309773,emotionally_stable,358.727273,0.363636,0.090909,0.022010,0.0,0.0,balanced
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,2,0.471495,emotionally_stable,632.666667,2.166667,0.375000,0.040376,0.0,0.0,balanced
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist,"{'positivity': 'high', 'verbosity': 'moderate'...",...,0,-0.263095,emotionally_stable,195.666667,1.333333,0.200000,0.042992,0.0,0.0,balanced
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer,"{'positivity': 'mixed', 'verbosity': 'moderate...",...,1,0.270200,emotionally_stable,406.545455,0.000000,0.000000,0.034632,0.0,0.0,balanced


In [97]:
# STEP 71 - View Reviews for a Specific User.. by user id

user_id = "-EX1hrPRBqNkVavtMllTCA"

user_reviews = curated_reviews[
    curated_reviews["user_id"] == user_id
].sort_values("date")

user_reviews[["date", "business_id", "stars", "text"]]

,date,business_id,stars,text
0,2011-12-15 12:02:27,agK5cXwnBQozM2M-5kLvzw,1,"We had food from this ""restaurant"" delivered t..."
1,2011-12-16 23:14:10,XnVmNQdmyhdCC90FRY9ZJQ,5,Had our Christmas party here this year. The s...
2,2012-02-04 17:10:19,GBTPC53ZrG1ZBY3DT8Mbcw,4,"Went here on a lark, basically. I had enough ..."
3,2012-02-07 18:27:15,-9yzQQ0d_rcOD2CzdTNO_Q,5,"This is one of the ""retro"" McDonald's and it's..."
4,2012-02-07 18:49:57,FEFi0AmjHzgSceeLYW3Glw,2,This location is inconsistent. Sometimes it b...
5,2012-02-09 19:06:19,HhD8DLES5ZeULKDXmfkOOg,5,I am currently listing to port as I type this....
6,2012-02-09 19:28:51,YUtYV7iXqiUX19gFBiy9SQ,2,Used to love to eat here when I was in high sc...
7,2012-02-16 22:17:35,C6YaSrMAzy3jJqinlFVudw,5,"If you're looking for excellent food, HUGE por..."
8,2012-02-21 22:02:19,jjMdu96R_7YV4qKsjRA2Gg,4,"I hit this location intermittently, usually wh..."
9,2012-02-23 19:39:16,G7G1QTVugC-jOwYkqs-UKg,1,Ate here all of once and that was enough.\n\nT...


In [98]:
# Print the reviews nicely

for _, row in user_reviews.iterrows():
    print(row["date"], row["stars"], row["business_id"])
    print(row["text"][:400])
    print("-" * 80)

2011-12-15 12:02:27 1 agK5cXwnBQozM2M-5kLvzw
We had food from this "restaurant" delivered to our office for a luncheon.  Big mistake.  Several orders screwed up.  Fried chicken and waffles were tasteless.  Tomato soup too salty.  And they managed to even screw up salads.  Imagine that.  Avoid like the plague.
--------------------------------------------------------------------------------
2011-12-16 23:14:10 5 XnVmNQdmyhdCC90FRY9ZJQ
Had our Christmas party here this year.  The service was top notch, the food came out right on time, and the steak was HEAVENLY.  The sides were excellent.  Definitely worth the trip!
--------------------------------------------------------------------------------
2012-02-04 17:10:19 4 GBTPC53ZrG1ZBY3DT8Mbcw
Went here on a lark, basically.  I had enough of C list luncheries and wanted something a bit better.  I ended up getting a Luke Burger and devoured it in moments.

The service was top notch.  My waiter, Richard, was always attentive and even made a gla

PHASE 4 is now complete with Linguistic Style Modeling. The personas now understand:

- storytelling behavior, emotional expressiveness, casual language, descriptive depth, communication styles

PHASE 5 — NIGERIAN CONTEXTUALIZATION

BUILD:
1. Create Nigerian linguistic markers
2. Detect local conversational style
3. Create cultural preference signals
4. Build Nigerian behavioral identities
5. Attach localized speaking styles to personas

In [ ]:
# STEP 72 — CREATE NIGERIAN EXPRESSION TAXONOMY
# This will allow us to capture cultural nuances in language use, which can provide deeper insights into user identities, 
# preferences, and sentiments in the context of Nigerian culture.

nigerian_expressions = {

    "soft_life": [
        "soft life", "premium enjoyment", "luxury vibes", "chill spot"
    ],

    "casual_slang": [
        "sha", "abi", "wahala", "dey", "no too bad", "pepper", "gist", "vibes", "omo", "no vex", "no gree for anybody"
    ],

    "pidgin_staples": [
        "abeg", "na wa o", "wahala", "sef", "nko", "abi", "na", "o",
        "ooh", "sha", "kpa", "kwa", "nawa", "mtchew", "chei", "chai",
        "walahi", "biko", "ndo", "jare", "gan", "sabi", "e choke"
    ],
    
    # Casual greetings & exclamations
    "exclamations": [
        "see finish", "see me see trouble", "oga", "madam", "boss",
        "my brother", "my sister", "my guy", "my dear", "babe",
        "sis", "bro", "chief", "alhaji", "mama", "papa"
    ],

    "social_enjoyment": [
        "hangout", "owambe", "groove", "turn up", "outing", "enjoyment" "it's giving", "vibes", "pepper", "gist", "chill spot"
    ],

    # Expressiveness & communication style (Pidgin, humour, directness)
    "expressiveness": [
        "abeg", "na wa o", "seems", "sef", "nko", "abi", "o", "ooh", "omo",
        "who send you", "na so so", "i no send your papa", "e enter"
        "walahi", "mtchew", "chai", "God willing", "not to praise am too much", "you dey whyne?"
    ],

    # Proverbs & wise sayings (often used to justify an opinion)
    "proverbs": [
        "a child who washes hands can eat with elders",
        "the lizard that jumps from a high tree would break its back",
        "when the music changes, the dance must change",
        "the one who throws a stone in the market forgets that others can throw too",
        "a bird that flies off the earth and lands on a tree is not safe from a stone",
        "he who brings kola brings life",
        "the way you dress is how you will be addressed",
        "it is not the size of the yam that matters, but the size of the stew",
        "a person who is chasing a rat cannot see the antelope",
        "if you want to hide a corpse, put it under a woman's wrapper"
    ],
    
    # Idiomatic expressions (figurative, not literal)
    "idioms": [
        "carry last",        # finish last / be embarrassed
        "chop breakfast",    # suffer a harsh disappointment
        "see finish",        # see someone's true colours / be fed up
        "form 419",          # act fraudulent or fake
        "blow grammar",      # speak overly fancy English
        "show pepper",       # be aggressive or tough
        "catch cruise",      # have fun / joke around
        "give attitude",     # behave rudely or arrogantly
        "carry go",          # take away / steal
        "use your head",     # think properly
        "shine your eye",    # be vigilant, don’t be fooled
        "do the needful",    # take necessary action
        "pull down",         # criticise or undermine someone
        "call somebody",     # confront or challenge
        "run mad",           # malfunction / go crazy
        "enter one chance",  # fall into a trap or irreversible situation
        "hot cake",          # very popular in demand
        "no gree for anybody" # stand your ground, don’t give in
    ],
    
    # Greetings & social expressions (used to open or close reviews)
    "greetings": [
        "how far?", "how now?", "how body?", "how market?",
        "hello o", "good morning o", "good afternoon o", "good evening o",
        "thank you jare", "thanks a lot", "appreciate",
        "sorry o", "my bad", "no wahala"
    ],
    
    # Exclamations & emotional outbursts (strong feelings)
    "exclamations": [
        "chai!", "chei!", "mtchew!", "no way!", "kpa!", "nawa o!",
        "God forbid!", "never!", "ehn?", "bawo?", "see glass!", "alas!!",
        "oga at the top!", "hallelujah!", "e shock you?", "e don happen!"
    ],
    
    # Figurative descriptions (vivid, often exaggerated)
    "figurative_descriptions": [
        "hot like suya",           # very hot
        "sweet like honey",        # delicious
        "bitter like agbo",        # very bitter
        "hard like rock",          # extremely hard/tough
        "soft like cotton",        # very soft
        "smooth like butter",      # very smooth
        "long like express",       # very long
        "slow like snail",         # extremely slow
        "fast like wind",          # very fast
        "small like ant",          # tiny
        "full like church on Sunday"  # very crowded
        "you dey whyne?" #are you joking?
    ],
    
    # Conditional & hypothetical phrases (storytelling markers)
    "conditional_phrases": [
        "if to say", "suppose say", "even if", "whether",
        "unless e be say", "as if", "imagine say", "make e be like say"
    ],
    
    # Persuasion & emphasis (used to convince reader)
    "persuasion": [
        "I swear down", "I swear for you", "believe me",
        "take it from me", "mark my word", "I guarantee you",
        "e no go better for you if you doubt", "try me"
    ],
    
    # Blame & criticism expressions
    "blame_criticism": [
        "the fault na", "na him cause am", "who send you?",
        "you no try", "e no correct", "wrong delivery",
        "na scam", "wayo", "419", "fake", "junk", "trash"
    ],
    
    # Humour & sarcasm markers
    "humour_sarcasm": [
        "I laff", "lolz", "mtchew", "see comedy", "joke of the year",
        "e be like film trick", "movie scene", "story for the gods",
        "you won't believe", "as if I never see", "everywhere first blur"
    ],
    
    # Cultural references (places, brands, events)
    "cultural_references": [
        "computer village",           # tech hub in Lagos
        "Alaba market",               # electronics market
        "Oshodi market",              # busy market
        "Balogun market",             # textile market
        "NEPA", "PHCN",               # electricity company
        "MTN", "Glo", "Airtel",       # network providers
        "BBNaija",                    # reality TV show
        "Nollywood",                  # film industry
        "Asake", "Burna Boy",         # popular musicians
        "Dangote",                    # conglomerate
        "Sallah", "Christmas",        # festivals
        "Ember months"                # September–December
    ],

    "practical_survival": [
        "traffic", "expensive", "affordable", "stress", "queue", "delay"
    ],

    "time_efficiency": [
        "wait time", "delay", "fast delivery", "slow", "African time",
        "hours", "minutes", "late", "early", "prompt", "wasted my time",
        "traffic", "Lagos traffic", "delivered on time"
    ],
    
    # Nigerian‑specific storytelling markers (colloquial narrative style)
    "naija_narrative": [
        "so I tell am", "the guy come say", "I just dey go",
        "immediately I enter", "see as e be", "before I know",
        "the next thing", "as I was coming", "I reach there",
        "the seller tell me", "my sister say make I try am",
        "story for another day", "I no fit shout", "you won't believe"
    ],
    
    # Exaggerated storytelling (typical of oral tradition)
    "hyperbole_narrative": [
        "I waited for years", "the longest hour of my life",
        "everybody in Lagos", "the whole market", "I almost died",
        "I swear down", "e be like film", "like a movie scene"
    ],


    # Kinship & relational terms (used to address or refer)
    "kinship": [
        "oga", "madam", "massa", "boss", "chief", "alhaji",
        "my brother", "my sister", "my guy", "my dear", "my friend",
        "uncle", "aunty", "papa", "mama", "baba", "iya", "bros"
    ],
    
    # Conjunctions & connectors (oral style flow)
    "oral_connectors": [
        "so I tell am", "immediately", "the next thing", "before I know",
        "as I dey go", "come see", "lo and behold", "to cut the long story short",
        "long story short", "in short", "and all that", "and so on"
    ]
}

In [100]:
# STEP 73 — CREATE NIGERIAN STYLE DETECTOR
# By counting the presence of Nigerian-specific expressions in reviews, we can identify users who tend to write in a Nigerian style, which may correlate with certain personas or sentiments.

def detect_nigerian_style(text, taxonomy):

    text = text.lower()

    scores = {}

    for category, phrases in taxonomy.items():

        score = 0

        for phrase in phrases:

            score += text.count(phrase)

        scores[category] = score

    return scores

In [101]:
# STEP 74 — APPLY NIGERIAN STYLE DETECTION
# This will allow us to identify the presence of Nigerian linguistic and storytelling elements in reviews, providing insights into cultural expression and identity among users.

curated_reviews["nigerian_style"] = (
    curated_reviews["text"]
    .apply(
        lambda x: detect_nigerian_style(
            x,
            nigerian_expressions
        )
    )
)

In [102]:
# STEP 75 — EXPAND NIGERIAN FEATURES

nigerian_style_df = pd.json_normalize(
    curated_reviews["nigerian_style"]
)

nigerian_style_df.head()

,soft_life,casual_slang,pidgin_staples,exclamations,social_enjoyment,expressiveness,proverbs,idioms,greetings,figurative_descriptions,...,persuasion,blame_criticism,humour_sarcasm,cultural_references,practical_survival,time_efficiency,naija_narrative,hyperbole_narrative,kinship,oral_connectors
0,0,0,17,0,0,16,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,8,0,0,8,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,19,0,0,19,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,13,0,0,12,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,25,0,0,24,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [103]:
# STEP 76 — MERGE NIGERIAN FEATURES

curated_reviews = pd.concat(
    [curated_reviews, nigerian_style_df],
    axis=1
)

curated_reviews.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,sentiment,...,persuasion,blame_criticism,humour_sarcasm,cultural_references,practical_survival,time_efficiency,naija_narrative,hyperbole_narrative,kinship,oral_connectors
0,6Vf1MkxDPrTcnuKYUldjFw,-EX1hrPRBqNkVavtMllTCA,agK5cXwnBQozM2M-5kLvzw,1,8,2,2,"We had food from this ""restaurant"" delivered t...",2011-12-15 12:02:27,-0.300000,...,0,0,0,0,0,0,0,0,0,0
1,Aninptga9OOZsmi5gFNAjQ,-EX1hrPRBqNkVavtMllTCA,XnVmNQdmyhdCC90FRY9ZJQ,5,0,0,0,Had our Christmas party here this year. The s...,2011-12-16 23:14:10,0.540179,...,0,0,0,0,0,0,0,0,0,0
2,EiPqGnP5SRapBaOkvZJAug,-EX1hrPRBqNkVavtMllTCA,GBTPC53ZrG1ZBY3DT8Mbcw,4,1,0,1,"Went here on a lark, basically. I had enough ...",2012-02-04 17:10:19,0.298437,...,0,0,0,0,0,0,0,0,0,0
3,pL3LWEwcqaTLaa7c4o860A,-EX1hrPRBqNkVavtMllTCA,-9yzQQ0d_rcOD2CzdTNO_Q,5,1,1,2,"This is one of the ""retro"" McDonald's and it's...",2012-02-07 18:27:15,0.276420,...,0,0,0,0,0,0,0,0,0,0
4,46Z1SVg6VJygnN3O95cFsw,-EX1hrPRBqNkVavtMllTCA,FEFi0AmjHzgSceeLYW3Glw,2,0,0,0,This location is inconsistent. Sometimes it b...,2012-02-07 18:49:57,-0.150000,...,0,0,0,0,0,1,0,0,0,0


In [104]:
# STEP 77 — AGGREGATE CULTURAL FEATURES PER USER
# This will allow us to aggregate the Nigerian linguistic and storytelling elements at the user level, 
# providing insights into individual cultural expression and identity.

nigerian_features = (
    curated_reviews
    .groupby("user_id")
    [
        list(nigerian_expressions.keys())
    ]
    .mean()
    .reset_index()
)

nigerian_features.head()

,user_id,soft_life,casual_slang,pidgin_staples,exclamations,social_enjoyment,expressiveness,proverbs,idioms,greetings,...,blame_criticism,humour_sarcasm,cultural_references,practical_survival,time_efficiency,time_efficiency,naija_narrative,hyperbole_narrative,kinship,oral_connectors
0,-EX1hrPRBqNkVavtMllTCA,0.0,0.166667,22.944444,0.0,0.111111,22.416667,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.055556,0.222222,0.333333,0.0,0.0,0.000000,0.000000
1,-M7fUg7FrdGctKr5f_eMUQ,0.0,0.136364,22.272727,0.0,0.045455,21.454545,0.0,0.0,0.045455,...,0.0,0.0,0.0,0.000000,0.227273,0.318182,0.0,0.0,0.000000,0.045455
2,-WM58wLjtlHlR91xVfM1FQ,0.0,0.250000,37.583333,0.0,0.000000,37.083333,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.250000,0.458333,0.0,0.0,0.041667,0.000000
3,-qTtg1D3RidRa4cTB-ftwg,0.0,0.000000,13.066667,0.0,0.000000,12.533333,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.066667,0.066667,0.0,0.0,0.000000,0.000000
4,02H49g16MdRoZKoX6IEoFA,0.0,0.181818,26.136364,0.0,0.000000,25.727273,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.136364,0.136364,0.0,0.0,0.000000,0.000000


In [105]:
# STEP 78 — MERGE INTO PERSONAS


persona_df = persona_df.merge(
    nigerian_features,
    on="user_id",
    how="left"
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile,...,blame_criticism,humour_sarcasm,cultural_references,practical_survival,time_efficiency_y,time_efficiency_y,naija_narrative,hyperbole_narrative,kinship,oral_connectors
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic,"{'positivity': 'low', 'verbosity': 'high', 'em...",...,0.0,0.0,0.0,0.055556,0.222222,0.333333,0.0,0.0,0.000000,0.000000
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,0.0,0.0,0.0,0.000000,0.227273,0.318182,0.0,0.0,0.000000,0.045455
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,0.0,0.0,0.0,0.000000,0.250000,0.458333,0.0,0.0,0.041667,0.000000
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist,"{'positivity': 'high', 'verbosity': 'moderate'...",...,0.0,0.0,0.0,0.000000,0.066667,0.066667,0.0,0.0,0.000000,0.000000
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer,"{'positivity': 'mixed', 'verbosity': 'moderate...",...,0.0,0.0,0.0,0.000000,0.136364,0.136364,0.0,0.0,0.000000,0.000000


In [106]:
# STEP 79 — CREATE NIGERIAN CULTURAL IDENTITIES

def nigerian_identity(row):

    if row["soft_life"] > 0.5:
        return "soft_life_explorer"

    elif row["casual_slang"] > 0.5:
        return "casual_slang"
    
    elif row["pidgin_staples"] > 0.5:
        return "pidgin_staples"
    
    elif row["exclamations"] > 0.5:
        return "exclamations"

    elif row["social_enjoyment"] > 0.5:
        return "social_enjoyment"
    
    elif row["expressiveness"] > 0.5:
        return "expressiveness"

    elif row["proverbs"] > 0.5:
        return "proverbs"

    elif row["greetings"] > 0.5:
        return "greetings"

    elif row["exclamations"] > 0.5:
        return "exclamations"

    elif row["figurative_descriptions"] > 0.5:
        return "figurative_descriptions"

    elif row["conditional_phrases"] > 0.5:
        return "conditional_phrases"

    elif row["persuasion"] > 0.5:
        return "persuasion"

    elif row["blame_criticism"] > 0.5:
        return "blame_criticism"

    elif row["humour_sarcasm"] > 0.5:
        return "humour_sarcasm"
    
    elif row["time_efficiency"] > 0.5:
        return "time_efficiency"

    elif row["naija_narrative"] > 0.5:
        return "naija_narrative"

    elif row["hyperbole_narrative"] > 0.5:
        return "hyperbole_narrative"

    elif row["oral_connectors"] > 0.5:
        return "oral_connectors"

    elif row["kinship"] > 0.5:
        return "kinship"
   
    elif row["practical_survival"] > 0.5:
        return "practical_survivor"


    else:
        return "globally_neutral"

In [107]:
persona_df["nigerian_identity"] = (
    persona_df.apply(
        nigerian_identity,
        axis=1
    )
)

persona_df[
    [
        "user_id",
        "archetype",
        "nigerian_identity"
    ]
].head(10)

,user_id,archetype,nigerian_identity
0,-EX1hrPRBqNkVavtMllTCA,Harsh Critic,pidgin_staples
1,-M7fUg7FrdGctKr5f_eMUQ,Emotional Storyteller,pidgin_staples
2,-WM58wLjtlHlR91xVfM1FQ,Emotional Storyteller,pidgin_staples
3,-qTtg1D3RidRa4cTB-ftwg,Warm Optimist,pidgin_staples
4,02H49g16MdRoZKoX6IEoFA,Reactive Reviewer,pidgin_staples
5,0650daOKAuufqymyOBe3cA,Warm Optimist,pidgin_staples
6,06Yz-YYYa1U9PN37b6UniA,Emotional Storyteller,pidgin_staples
7,0Tsu6-uhw_w9Z3W0Xlnazg,Harsh Critic,pidgin_staples
8,0xJwTzZuWOac7ufeTioeig,Emotional Storyteller,pidgin_staples
9,175DuOm7IPiEy5f1KG9esA,Harsh Critic,pidgin_staples


FINAL PHASE OF LAYER 1
PHASE 6 — PERSONA SUMMARIZATION & COGNITIVE PROFILES

BUILD:
- Create persona narratives.
- Synthesize traits.
- Create recommendation tendencies.
- Create human-readable summaries.
- Build cognitive identity profiles.

In [108]:
# STEP 80 — CREATE PERSONA SUMMARY FUNCTION
# This will allow us to generate a concise summary of each user persona.
# By summarizing the key characteristics of each persona, including their archetype, 
# communication style, dominant values, emotional trajectory, and Nigerian identity, 
# we can create a more holistic and human-readable profile for each user.

def generate_persona_summary(row):

    summary = f"""
    Archetype: {row['archetype']}

    Communication Style:
    {row['communication_style']}

    Dominant Value:
    {row['dominant_value']}

    Emotional Trajectory:
    {row['emotional_trajectory']}

    Nigerian Identity:
    {row['nigerian_identity']}
    """

    return summary.strip()

In [109]:
# STEP 81 — GENERATE PERSONA SUMMARIES
# This will create a human-readable summary for each user persona, encapsulating their key characteristics and insights in a concise format.


persona_df["persona_summary"] = (
    persona_df.apply(
        generate_persona_summary,
        axis=1
    )
)

persona_df[
    [
        "user_id",
        "persona_summary"
    ]
].head()

,user_id,persona_summary
0,-EX1hrPRBqNkVavtMllTCA,Archetype: Harsh Critic\n\n Communication S...
1,-M7fUg7FrdGctKr5f_eMUQ,Archetype: Emotional Storyteller\n\n Commun...
2,-WM58wLjtlHlR91xVfM1FQ,Archetype: Emotional Storyteller\n\n Commun...
3,-qTtg1D3RidRa4cTB-ftwg,Archetype: Warm Optimist\n\n Communication ...
4,02H49g16MdRoZKoX6IEoFA,Archetype: Reactive Reviewer\n\n Communicat...


In [110]:
# STEP 83 — APPLY RECOMMENDATION TENDENCIES

persona_df["recommendation_tendency"] = (
    persona_df.apply(
        recommendation_tendency,
        axis=1
    )
)

persona_df[
    [
        "user_id",
        "recommendation_tendency"
    ]
].head()

,user_id,recommendation_tendency
0,-EX1hrPRBqNkVavtMllTCA,prioritizes authentic and high-quality meals
1,-M7fUg7FrdGctKr5f_eMUQ,"expects courteous, attentive, and reliable staff"
2,-WM58wLjtlHlR91xVfM1FQ,prioritizes authentic and high-quality meals
3,-qTtg1D3RidRa4cTB-ftwg,balanced preferences
4,02H49g16MdRoZKoX6IEoFA,"prefers easy access, fast delivery, and hassle..."


In [111]:
# STEP 84 — CREATE FULL COGNITIVE PROFILE
# Now we synthesize the complete behavioral human.

def build_cognitive_profile(row):

    profile = {

        "archetype": row["archetype"],

        "communication_style":
            row["communication_style"],

        "dominant_value":
            row["dominant_value"],

        "emotional_trajectory":
            row["emotional_trajectory"],

        "nigerian_identity":
            row["nigerian_identity"],

        "recommendation_behavior":
            row["recommendation_tendency"]
    }

    return profile

In [112]:
persona_df["cognitive_profile"] = (
    persona_df.apply(
        build_cognitive_profile,
        axis=1
    )
)

persona_df[
    [
        "user_id",
        "cognitive_profile"
    ]
].head(15)

,user_id,cognitive_profile
0,-EX1hrPRBqNkVavtMllTCA,"{'archetype': 'Harsh Critic', 'communication_s..."
1,-M7fUg7FrdGctKr5f_eMUQ,"{'archetype': 'Emotional Storyteller', 'commun..."
2,-WM58wLjtlHlR91xVfM1FQ,"{'archetype': 'Emotional Storyteller', 'commun..."
3,-qTtg1D3RidRa4cTB-ftwg,"{'archetype': 'Warm Optimist', 'communication_..."
4,02H49g16MdRoZKoX6IEoFA,"{'archetype': 'Reactive Reviewer', 'communicat..."
5,0650daOKAuufqymyOBe3cA,"{'archetype': 'Warm Optimist', 'communication_..."
6,06Yz-YYYa1U9PN37b6UniA,"{'archetype': 'Emotional Storyteller', 'commun..."
7,0Tsu6-uhw_w9Z3W0Xlnazg,"{'archetype': 'Harsh Critic', 'communication_s..."
8,0xJwTzZuWOac7ufeTioeig,"{'archetype': 'Emotional Storyteller', 'commun..."
9,175DuOm7IPiEy5f1KG9esA,"{'archetype': 'Harsh Critic', 'communication_s..."


In [113]:
# STEP 85 — MANUAL INSPECTION
# Inspect several cognitive profiles.

persona_df[
    [
        "user_id",
        "cognitive_profile"
    ]
].sample(5)

,user_id,cognitive_profile
135,RscoPhnfz9HFXN7X8czFlg,"{'archetype': 'Warm Optimist', 'communication_..."
176,brwx3_ZbrER7fFHWWKutmQ,"{'archetype': 'Reactive Reviewer', 'communicat..."
140,U8RoyRIRrmsxHz-A0IBz3A,"{'archetype': 'Warm Optimist', 'communication_..."
75,Hq_iRg_7i-nl_LFuGrv6Kg,"{'archetype': 'Deep Experience Analyst', 'comm..."
271,v12_ycZ0eSjtkirSWuON5w,"{'archetype': 'Emotional Storyteller', 'commun..."


In [114]:
# View one full cognitive profile

persona_df.iloc[17]["cognitive_profile"]

{'archetype': 'Harsh Critic',
 'communication_style': 'balanced',
 'dominant_value': 'food_quality',
 'emotional_trajectory': 'emotionally_stable',
 'nigerian_identity': 'pidgin_staples',
 'recommendation_behavior': 'prioritizes authentic and high-quality meals'}

In [115]:
# OR

import pprint

pp = pprint.PrettyPrinter(indent=4)

pp.pprint(
    persona_df.iloc[17]["cognitive_profile"]
)

{   'archetype': 'Harsh Critic',
    'communication_style': 'balanced',
    'dominant_value': 'food_quality',
    'emotional_trajectory': 'emotionally_stable',
    'nigerian_identity': 'pidgin_staples',
    'recommendation_behavior': 'prioritizes authentic and high-quality meals'}


In [116]:
# OR
 
cognitive_expanded = pd.json_normalize(
    persona_df["cognitive_profile"]
)

cognitive_expanded.head()

,archetype,communication_style,dominant_value,emotional_trajectory,nigerian_identity,recommendation_behavior
0,Harsh Critic,balanced,food_quality,emotionally_stable,pidgin_staples,prioritizes authentic and high-quality meals
1,Emotional Storyteller,balanced,service_quality,emotionally_stable,pidgin_staples,"expects courteous, attentive, and reliable staff"
2,Emotional Storyteller,balanced,food_quality,emotionally_stable,pidgin_staples,prioritizes authentic and high-quality meals
3,Warm Optimist,balanced,positive_exaggeration,emotionally_stable,pidgin_staples,balanced preferences
4,Reactive Reviewer,balanced,convenience,emotionally_stable,pidgin_staples,"prefers easy access, fast delivery, and hassle..."


In [117]:
# Pick a user

user_id = persona_df.iloc[4]["user_id"]

# Show profile
pp.pprint(
    persona_df.iloc[0]["cognitive_profile"]
)

# Get reviews
user_reviews = curated_reviews[
    curated_reviews["user_id"] == user_id
]

# Display reviews
user_reviews[
    ["stars", "text"]
].head(3)

{   'archetype': 'Harsh Critic',
    'communication_style': 'balanced',
    'dominant_value': 'food_quality',
    'emotional_trajectory': 'emotionally_stable',
    'nigerian_identity': 'pidgin_staples',
    'recommendation_behavior': 'prioritizes authentic and high-quality meals'}


,stars,text
97,4,I come here almost every Saturday after the gy...
98,5,The best Cambodian food in Philly.\n\nThe serv...
99,4,I came here for a birthday dinner. We had a p...


Layer 1 is essentially complete. The persona engine and system now understands:

- personality archetypes
- values
- emotional behavior
- temporal evolution
- changing tastes
- communication style
- Nigerian contextual behavior
- recommendation tendencies

That is HUMAN UNDERSTANDING.

In [118]:
# SAVING PERSONA PROFILES

persona_df.to_csv(
    "../data/processed/persona_profiles.csv",
    index=False
)

In [119]:
persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile,...,time_efficiency_y,time_efficiency_y,naija_narrative,hyperbole_narrative,kinship,oral_connectors,nigerian_identity,persona_summary,recommendation_tendency,cognitive_profile
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic,"{'positivity': 'low', 'verbosity': 'high', 'em...",...,0.222222,0.333333,0.0,0.0,0.000000,0.000000,pidgin_staples,Archetype: Harsh Critic\n\n Communication S...,prioritizes authentic and high-quality meals,"{'archetype': 'Harsh Critic', 'communication_s..."
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,0.227273,0.318182,0.0,0.0,0.000000,0.045455,pidgin_staples,Archetype: Emotional Storyteller\n\n Commun...,"expects courteous, attentive, and reliable staff","{'archetype': 'Emotional Storyteller', 'commun..."
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,0.250000,0.458333,0.0,0.0,0.041667,0.000000,pidgin_staples,Archetype: Emotional Storyteller\n\n Commun...,prioritizes authentic and high-quality meals,"{'archetype': 'Emotional Storyteller', 'commun..."
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist,"{'positivity': 'high', 'verbosity': 'moderate'...",...,0.066667,0.066667,0.0,0.0,0.000000,0.000000,pidgin_staples,Archetype: Warm Optimist\n\n Communication ...,balanced preferences,"{'archetype': 'Warm Optimist', 'communication_..."
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer,"{'positivity': 'mixed', 'verbosity': 'moderate...",...,0.136364,0.136364,0.0,0.0,0.000000,0.000000,pidgin_staples,Archetype: Reactive Reviewer\n\n Communicat...,"prefers easy access, fast delivery, and hassle...","{'archetype': 'Reactive Reviewer', 'communicat..."


In [120]:
# SAVING CURATED REVIEWS WITH ENRICHED FEATURES

curated_reviews.to_csv(
    "../data/processed/curated_reviews.csv",
    index=False
)

In [121]:
# SAVING BUSINESS

business_df.to_csv(
    "../data/processed/businesses.csv",
    index=False
)

In [122]:
# SAVING USERS
from src.personas.archetypes import (
    prepare_clustering_features,
    scale_features,
    run_kmeans_clustering,
    assign_archetypes
)